# Main workflow — parameter estimation and full-dimensional Bayesian inference

This notebook runs the complete analysis of one experiment (one tr-PL decay, optionally ss-PL data)
with the SRH model with detrapping (1 or 2 traps) and the neural-network surrogate.
It replaces the pairwise-sliced grid posterior of the original submission.

| Section | What happens | Always / optional | Plan task |
|---|---|---|---|
| 0 | settings, imports of my modules (`preprocessing/`) and of the `inference` package | always | |
| 1 | load the NN surrogate; prior = training bounds of the NN (+ $E_{t1}<E_{t2}$) | always | T14 |
| 2 | **import data**: 2a tr-PL, 2b ss-PL | always | |
| 3 | **preprocessing**: 3a tr-PL (trim → spline → $\tau_\mathrm{diff}$ vs QFLS), 3b ss-PL (units) | optional / always | |
| 4 | likelihood — the *single* objective used by every later step; throughput benchmark | always | T8 |
| 5 | **start-point estimation** (physics-informed, legacy estimator) | always | T2, T8 |
| 6 | **fitting**: CMA-ES (NN) → `curve_fit` with the full ODE; $\sigma_\mathrm{eff}$; exports (.dat, ODE and NN curves) | always | T8 |
| 7 | **inference**: NUTS (TFP) + emcee, diagnostics, credible intervals; 7f full-dimensional **grid**; 7g ODE re-weighting (PSIS); 7h trap dynamics across the posterior | always / optional | T1, T4, T7, T9 |
| 8 | sensitivity to the start values (optimiser incl. ODE refinement, and chains) | optional | T2 |
| 9 | profile likelihood (all other parameters re-optimised) | optional | T3 |
| 10 | comparison tr-PL-only vs joint run | optional | T4 |
| 11 | timing table incl. offline costs (simulations, training) and ODE estimates | always | T8 |

The posterior of the publication is sampled with tempered SMC in `smc.ipynb` (the configs set `nuts.run` and
`emcee.run` to false); `ppc.ipynb` checks it and `ode_inference.ipynb` compares it with the rate equations.
Simulation-based calibration: `scripts/sbc_smc.py`.

Every step **plots** its result and **exports** it to `results/<run_name>_<mode>/` (CSV/JSON + `figures/`).
All fits and the sampling are timed with `timeit` (`inference.benchmark.timeit_call`) and the number of
likelihood evaluations is counted (`inference.benchmark.EvalCounter`, also inside XLA-compiled code).

**How to use**
1. Choose the experiment (`CONFIG_PATH`) and the mode (`'trpl'`, `'sspl'` or `'joint'`) in Section 0.
2. Switch the optional sections on/off in Section 0 (or in the config file).
3. Run all cells.  `QUICK = True` gives a fast smoke test with small sample sizes (not for results).

**Environment**: TensorFlow 2.21, TensorFlow Probability 0.25 (+ tf_keras), emcee 3.1, ArviZ 1.3, pycma
(see `requirements.txt`).  The two trained networks are in `networks/1trap/` and `networks/2trap/`; the
config files point to them.

## 0. Settings and imports

In [ ]:
# ── Which experiment ─────────────────────────────────────────────────────────
# One JSON file per experiment (folder configs/).  All paths inside are
# relative to the repository root (= the folder of this notebook).
CONFIG_PATH = "configs/2trap_85-15_film_inference.json"

# Inference mode: 'trpl' (tr-PL only), 'sspl' (ss-PL only: exact ss-PL model, no surrogate)
# or 'joint' (tr-PL + ss-PL).
# None = use the value in the config.  For the comparison in Section 10 run
# the notebook once with each mode.
MODE = None

# ── Optional sections (None = use the config value) ──────────────────────────
DO_PREPROCESS_TRPL = None   # raw transient -> spline -> tau_diff vs QFLS (Section 3a)
RUN_NUTS           = None   # Section 7a: NUTS posterior; False = stop after the fits (posterior with smc.ipynb)
RUN_EMCEE          = None   # emcee cross-check of NUTS (Section 7b)
RUN_SENSITIVITY    = None   # Section 8: sensitivity to the start values (T2)
RUN_PROFILE        = None   # Section 9: profile likelihood (T3)
RUN_ODE_CHECK      = None   # compare the best fit with the rate equations (Section 6c)
RUN_ODE_FIT        = None   # CMA-ES with the ODE instead of the NN — slow, timing table only (6f)
RUN_ODE_REFINE     = None   # after CMA-ES: scipy curve_fit with the full ODE (Section 6)
RUN_GRID           = None   # Section 7f: full-dimensional grid inference
GRID_POINTS_PER_AXIS = None # grid resolution: points per parameter axis (cost = n^D evaluations!); None = config value
GRID_RANGE         = None   # 'window' (around the physics-informed fit) or 'prior' (full box)
RUN_ODE_REWEIGHT   = None   # Section 7g: ODE re-evaluation of posterior samples + PSIS (T7)
RUN_TRAP_DYNAMICS  = None   # Section 7h: trap dynamics across the posterior (T9, 2 traps)

# ── Timing (Section 11): timeit repeats -> mean ± SD in the T8 table ────────────
TIMING_REPEATS      = None  # short steps: preprocessing, start values, fits, curve_fit (None = config, default 5)
TIMING_REPEATS_LONG = None  # NUTS, emcee, grid, ODE-only CMA-ES — every repeat is a FULL run (None = config, default 3)
NO_TIMING = False           # 24/10/2026 (Robin): every timed step runs ONCE (no warm-up, no repeats, no log-L benchmark)

# Folder of a finished run of the OTHER mode for Section 10 (or None)
COMPARE_WITH = None         # e.g. "results/2trap_85-15_film/2trap_85-15_film_trpl"

# Quick smoke test: small sample sizes everywhere (NOT for results)
QUICK = False

# GPU (must be set before TensorFlow is imported); None = config value
CUDA_DEVICE = None

In [ ]:
import os, io, json, copy, contextlib, warnings
from pathlib import Path

with open(CONFIG_PATH) as fh:
    cfg = json.load(fh)

# Environment for TensorFlow — must happen before the first TF import
os.environ["CUDA_VISIBLE_DEVICES"] = str(CUDA_DEVICE if CUDA_DEVICE is not None
                                         else cfg["hardware"]["cuda_device"])
os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "2")   # hide TF info logs (XLA prints a spurious memory estimate)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
%matplotlib inline
warnings.filterwarnings("ignore", category=UserWarning)        # torch / sklearn version chatter
warnings.filterwarnings("ignore", category=DeprecationWarning)

# ── My modules from the original workflow (preprocessing/) ───────────────────
import inference
inference.add_legacy_paths()                 # puts preprocessing/ on sys.path
import data_utils                            # tr-PL preprocessing: trim, spline, (t, PL) -> (QFLS, tau_diff)
import start_point_estimation_module as spe  # physics-informed start values (used via inference.start_points)

# ── New inference package (inference/) ───────────────────────────────────────
from inference import (surrogate, sspl_tf, likelihood, data, start_points, optimize, refine,
                       samplers, grid, validate, dynamics, profile, ode, report)
from inference.prior import Prior
from inference import benchmark
from inference.benchmark import timeit_call, count_evals, benchmark_log_post

print("TensorFlow", tf.__version__, "| GPUs:", [g.name for g in tf.config.list_physical_devices("GPU")])

In [ ]:
# ── Resolve the settings (notebook value overrides config) ───────────────────
def pick(nb_value, cfg_value):
    return cfg_value if nb_value is None else nb_value

MODE               = pick(MODE, cfg["mode"])
DO_PREPROCESS_TRPL = pick(DO_PREPROCESS_TRPL, cfg["trpl"]["preprocess"])
RUN_NUTS           = pick(RUN_NUTS, cfg["nuts"].get("run", True))
RUN_EMCEE          = pick(RUN_EMCEE, cfg["emcee"]["run"])
RUN_SENSITIVITY    = pick(RUN_SENSITIVITY, cfg["sensitivity"]["run"])
RUN_PROFILE        = pick(RUN_PROFILE, cfg["profile"]["run"])
RUN_ODE_CHECK      = pick(RUN_ODE_CHECK, cfg["ode"]["check_best_fit"])
RUN_ODE_FIT        = pick(RUN_ODE_FIT, cfg["ode"]["run_ode_fit"])
RUN_ODE_REFINE     = pick(RUN_ODE_REFINE, cfg["fit"].get("ode_refine", True))
RUN_GRID           = pick(RUN_GRID, cfg["grid"]["run"])
GRID_POINTS_PER_AXIS = pick(GRID_POINTS_PER_AXIS, cfg["grid"]["points_per_axis"])
GRID_RANGE         = pick(GRID_RANGE, cfg["grid"]["range"])
RUN_ODE_REWEIGHT   = pick(RUN_ODE_REWEIGHT, cfg["t7_ode_reweight"]["run"])
RUN_TRAP_DYNAMICS  = pick(RUN_TRAP_DYNAMICS, cfg["t9_trap_dynamics"]["run"])
TIMING_REPEATS      = int(pick(TIMING_REPEATS, cfg.get("timing", {}).get("repeats", 5)))
TIMING_REPEATS_LONG = int(pick(TIMING_REPEATS_LONG, cfg.get("timing", {}).get("repeats_long", 3)))
if NO_TIMING:      # single runs: the T8 table then holds single times without SD
    TIMING_REPEATS = TIMING_REPEATS_LONG = 1
WARMUP = 0 if NO_TIMING else 1   # untimed warm-up calls before the timed repeats
HAVE_SSPL          = bool(cfg["sspl"].get("use", False))
if not RUN_NUTS:   # everything below needs NUTS samples (posterior then from smc.ipynb)
    RUN_EMCEE = RUN_GRID = RUN_ODE_REWEIGHT = RUN_TRAP_DYNAMICS = RUN_SENSITIVITY = RUN_PROFILE = False
    COMPARE_WITH = None
    print("RUN_NUTS = False: sections 7-10 are skipped (emcee, grid, 7g, 7h, 8, 9, 10); run smc.ipynb for the posterior")
if MODE not in ("trpl", "sspl", "joint"):
    raise ValueError("MODE must be 'trpl', 'sspl' or 'joint'")
if MODE in ("sspl", "joint") and not HAVE_SSPL:
    raise ValueError(f"MODE='{MODE}' needs ss-PL data (sspl.use = true in the config)")

if QUICK:   # small sizes for a fast end-to-end test of the code paths
    cfg["nuts"].update(n_chains=8, n_warmup=300, n_draws=300)
    cfg["emcee"].update(n_steps=3000)
    cfg["fit"].update(max_evals=3000, n_extra_sobol_starts=3)
    cfg["sensitivity"].update(n_sobol_starts=8, max_evals=3000, n_chains_per_group=4, n_warmup=200, n_draws=200)
    cfg["profile"].update(n_grid=10, n_random=2, n_sweeps=1)
    cfg["fit"].update(ode_max_nfev=15)
    cfg["t7_ode_reweight"].update(n_samples=100)
    cfg["t9_trap_dynamics"].update(n_samples=50)
    GRID_POINTS_PER_AXIS = min(GRID_POINTS_PER_AXIS, 4)
    TIMING_REPEATS, TIMING_REPEATS_LONG = min(TIMING_REPEATS, 2), min(TIMING_REPEATS_LONG, 2)

# Physics of the sample
Eg     = cfg["physics"]["Eg_eV"]          # bandgap [eV]
npulse = cfg["physics"]["npulse_cm3"]     # excitation density at t = 0 [cm^-3]
T      = cfg["physics"]["T_K"]
d_cm   = cfg["physics"]["thickness_nm"] * 1e-7

# Output folders
RUN_DIR = Path(cfg["output_dir"]) / f"{cfg['run_name']}_{MODE}{'_QUICK' if QUICK else ''}"
FIG_DIR = RUN_DIR / "figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)
report.save_json({"config_path": CONFIG_PATH, "mode": MODE, "quick": QUICK,
                  "switches": dict(DO_PREPROCESS_TRPL=DO_PREPROCESS_TRPL, RUN_NUTS=RUN_NUTS, RUN_EMCEE=RUN_EMCEE,
                                   RUN_SENSITIVITY=RUN_SENSITIVITY, RUN_PROFILE=RUN_PROFILE,
                                   RUN_ODE_CHECK=RUN_ODE_CHECK, RUN_ODE_FIT=RUN_ODE_FIT,
                                   RUN_ODE_REFINE=RUN_ODE_REFINE, RUN_GRID=RUN_GRID,
                                   GRID_POINTS_PER_AXIS=GRID_POINTS_PER_AXIS, GRID_RANGE=GRID_RANGE,
                                   RUN_ODE_REWEIGHT=RUN_ODE_REWEIGHT, RUN_TRAP_DYNAMICS=RUN_TRAP_DYNAMICS,
                                   TIMING_REPEATS=TIMING_REPEATS, TIMING_REPEATS_LONG=TIMING_REPEATS_LONG),
                  "config": cfg}, RUN_DIR / "settings.json")

# Collector for the timing table of Section 11 (one row per timed step)
TIMINGS = []
def log_timing(timing, n_evals=None, n_grad_evals=None, note=""):
    """Store a timeit result (+ evaluation counts) for the summary table."""
    row = timing.as_dict()
    row.update(n_evals=n_evals, n_grad_evals=n_grad_evals, note=note)
    TIMINGS.append(row)
    print(timing, f"| evaluations: {n_evals}" if n_evals is not None else "",
          f"| with gradient: {n_grad_evals}" if n_grad_evals is not None else "")

print(f"Run: {cfg['run_name']}  mode={MODE}  -> {RUN_DIR}")

## 1. Neural-network surrogate and prior

* The network maps $[\log_{10}n_\mathrm{pulse}, E_g, \log_{10}k_\mathrm{rad}, (\log_{10}N_t, E_t/E_g, \log_{10}\tau_n, \log_{10}\tau_p)_k]$
  to 256 values of $\log_{10}\tau_\mathrm{diff}$ on a fixed QFLS axis.
* **Prior** = uniform on the *training box* of the network (prior = optimisation = sampling = training
  bounds, R1 comment 6 / Table S1).  For two traps the energies are ordered, $E_{t1}<E_{t2}$: this removes
  the mirror modes, and the 2-trap training data contain *only* ordered energies.
* Fixed parameters (config `prior.fixed`, e.g. $k_\mathrm{rad}=10^{-11}$ cm$^3$/s) are held constant everywhere
  (fit, sampling, grid) and are left out of the plots.
* The train/val/test split sizes printed here are the numbers for R2 comment 3c.
* QFLS-axis check (networks before 10/2026): training curves were interpolated on their own QFLS axis; the surrogate
  assumes the analytic axis of $(n_\mathrm{pulse}, E_g)$.  The printed fractions tell how often (and how often near this
  experiment) the two differ by more than 1 meV at the upper end / 10 meV at the lower end.
* PL-decade networks (from 10/2026, `surrogate_meta.json` in the network folder): the targets live on the fixed axis
  $u=\log_{10}(\mathrm{PL}/\mathrm{PL}_0)\in[-12,0]$ (excess-density ODE from $t=0$, every training curve spans the
  full 12 decades, invalid points masked).  The measured curve is compared on the same $u$ (exact inverse of the
  $p_0=0$ calibration of the preprocessing), so $p_0$ from trap neutrality never enters the likelihood.  The check
  prints how many simulated curves were removed in the preparation.

In [ ]:
nn_dir, stem = Path(cfg["nn"]["dir"]), cfg["nn"].get("stem")
art = surrogate.load_network(nn_dir, stem)      # PL-decade networks: paths from surrogate_meta.json
N_TRAPS = art.n_traps
print(f"Network: {nn_dir.name}  ({N_TRAPS} trap(s), {art.n_inputs} inputs)")
print("Train / val / test split:", art.split_sizes, "-> total", sum(art.split_sizes.values()))
print(f"Training range: log10 n_pulse = {art.lb[0]:.4f} … {art.ub[0]:.4f},  Eg = {art.lb[1]} … {art.ub[1]} eV")

# Training curves were interpolated on their own axis linspace(q_lo, q_hi, 256).  Networks without axis head assume
# ONE analytic axis per (n_pulse, Eg); report how many training curves deviate (generalised networks: a sizeable
# fraction, fixed-n_pulse/Eg networks: ~0.5 %).  Axis-head networks (art.axis == "predicted_window") predict
# q_max and the span per theta (nn-training/train_axis_head.py), so this check only describes the training data.
print("Network QFLS axis:", art.axis)
axis_check = surrogate.qfls_axis_consistency(art, npulse, Eg)
print("QFLS-axis check of the training curves:", {k: round(v, 4) if isinstance(v, float) else v
                                                   for k, v in axis_check.items()})
report.save_json({"network": str(nn_dir), "split_sizes": art.split_sizes, "qfls_axis_check": axis_check,
                  "lb": art.lb, "ub": art.ub, "param_names": art.param_names}, RUN_DIR / "network_info.json")

# Prior on the training box (+ ordering / optional photodoping constraint)
prior = Prior.from_artifacts(art, Eg=Eg, order_traps=cfg["prior"]["order_traps"],
                             photodoping_traps=cfg["prior"]["photodoping_traps"],
                             bounds_override=cfg["prior"]["bounds_override"],
                             fixed=cfg["prior"].get("fixed", {}), T=T)
D = prior.dim
bounds_table = prior.describe()
print("Constraints:", bounds_table.attrs["constraints"])
bounds_table.to_csv(RUN_DIR / "prior_bounds.csv", index=False)
bounds_table

## 2. Import data
### 2a. tr-PL
Either a raw transient (time [ns], PL) or an already preprocessed file (QFLS, $\tau_\mathrm{diff}$).

In [ ]:
tr_cfg = cfg["trpl"]
if tr_cfg["format"] == "raw":
    # index: time [ns], column 'pl'; skiprows='auto' skips header lines only
    trpl_raw = data.load_trpl_raw(tr_cfg["path"], skiprows=tr_cfg.get("skiprows", "auto"))
    trpl_processed_in = None
    print(f"Raw transient: {len(trpl_raw)} points, t = {trpl_raw.index.min():.3g} … {trpl_raw.index.max():.3g} ns")
else:
    trpl_raw = None
    trpl_processed_in = data.load_trpl_processed(tr_cfg["path"])   # columns qfls, tau_diff
    print(f"Preprocessed tr-PL: {len(trpl_processed_in)} points, QFLS = "
          f"{trpl_processed_in.qfls.min():.3f} … {trpl_processed_in.qfls.max():.3f} eV")

fig, ax = plt.subplots(figsize=(5, 3.5))
if trpl_raw is not None:
    ax.loglog(trpl_raw.index, trpl_raw["pl"], "o", ms=3)
    ax.set_xlabel("time [ns]"); ax.set_ylabel("PL [a.u.]")
else:
    ax.semilogy(trpl_processed_in.qfls, trpl_processed_in.tau_diff, "o", ms=3)
    ax.set_xlabel("QFLS [eV]"); ax.set_ylabel(r"$\tau_\mathrm{diff}$ [s]")
ax.set_title("tr-PL data as imported"); ax.grid(alpha=0.3)
report.save_figure(fig, FIG_DIR / "2a_trpl_imported.png"); plt.show()

### 2b. ss-PL
Standard columns after import: `intensity_suns`, `qfls` [eV], `plqy` (fraction), `G_cm3s` (whichever exist).

In [ ]:
ss_cfg = cfg["sspl"]
if HAVE_SSPL:
    sspl_df = data.load_sspl(ss_cfg["path"], fmt=ss_cfg["format"], plqy_in_percent=ss_cfg["plqy_in_percent"])
    print(f"ss-PL: {len(sspl_df)} points, columns {list(sspl_df.columns)}")
    display(sspl_df.head())
    fig, ax = plt.subplots(figsize=(5, 3.5))
    xc = "intensity_suns" if "intensity_suns" in sspl_df else "qfls"
    for col, mk in (("qfls", "o"), ("plqy", "s")):
        if col in sspl_df and col != xc:
            ax.plot(sspl_df[xc], sspl_df[col], mk, label=col)
    ax.set_xscale("log" if xc == "intensity_suns" else "linear"); ax.set_xlabel(xc)
    ax.legend(); ax.grid(alpha=0.3); ax.set_title("ss-PL data as imported")
    report.save_figure(fig, FIG_DIR / "2b_sspl_imported.png"); plt.show()
else:
    sspl_df = None
    print("No ss-PL data used (sspl.use = false).")

## 3. Preprocessing
### 3a. tr-PL (optional): trim → normalise → monotone spline → $\tau_\mathrm{diff}$ vs QFLS
Uses my functions in `preprocessing/data_utils.py` exactly as in the original Figure notebooks:
`trim_and_normalize` (start at the PL peak, PL$_\max$ = 1), `spline_torch` (monotone cubic B-spline in
log–log space) and `t_pl_to_qfls_tau` (QFLS $= V_t\ln(\mathrm{PL}/\mathrm{PL}_\max) + 2V_t\ln(n_\mathrm{pulse}/n_i)$,
$\tau_\mathrm{diff} = -2/(\mathrm{d}\ln\mathrm{PL}/\mathrm{d}t)$).  Afterwards non-finite points and $\tau_\mathrm{diff}\le0$ are removed.

In [ ]:
if DO_PREPROCESS_TRPL:
    if trpl_raw is None:
        raise ValueError("Preprocessing needs the raw transient (trpl.format = 'raw').")

    RAW_POINTS = tr_cfg.get("points", "spline") == "raw"
    if RAW_POINTS:
        # trpl.points = "raw" (Robin, 20/10/2026): the measured points converted directly, no spline fit
        def preprocess_trpl():
            trimmed = data_utils.trim_and_normalize(trpl_raw.copy())
            return trimmed, None, data_utils.t_pl_to_qfls_tau(trimmed, npulse, Eg)
    else:
        def preprocess_trpl():
            trimmed = data_utils.trim_and_normalize(trpl_raw.copy())
            with contextlib.redirect_stdout(io.StringIO()):          # spline_torch prints every epoch
                spline = data_utils.spline_torch(trimmed, n_points=tr_cfg["spline_knots"], degree=3,
                                                 n_pred=tr_cfg["spline_points"])
            return trimmed, spline, data_utils.t_pl_to_qfls_tau(spline, npulse, Eg)

    (trpl_trimmed, trpl_spline, trpl_processed), t_pre = timeit_call(preprocess_trpl, repeats=TIMING_REPEATS, warmup=WARMUP,
                                                                 label="tr-PL preprocessing")
    log_timing(t_pre)
    trpl_raw_conv = data_utils.t_pl_to_qfls_tau(trpl_trimmed, npulse, Eg)   # raw points, for the plot only

    # check plots: transient + spline, tau_diff vs QFLS
    fig, ax = plt.subplots(1, 2, figsize=(10, 3.8))
    ax[0].loglog(trpl_trimmed.index, trpl_trimmed["pl"], "o", ms=3, label="data (trimmed)")
    if trpl_spline is not None:
        ax[0].loglog(trpl_spline.index, trpl_spline["pl"], "-", lw=2, label="spline")
    ax[0].set_xlabel("time [ns]"); ax[0].set_ylabel("PL (norm.)"); ax[0].legend(); ax[0].grid(alpha=0.3)
    ax[1].semilogy(trpl_raw_conv.qfls, trpl_raw_conv.tau_diff, "o", ms=3, alpha=0.5, label="raw data")
    if trpl_spline is not None:
        ax[1].semilogy(trpl_processed.qfls, trpl_processed.tau_diff, "-", lw=2, label="spline")
    ax[1].set_xlabel("QFLS [eV]"); ax[1].set_ylabel(r"$\tau_\mathrm{diff}$ [s]"); ax[1].legend(); ax[1].grid(alpha=0.3)
    fig.tight_layout(); report.save_figure(fig, FIG_DIR / "3a_trpl_preprocessing.png"); plt.show()
    trpl_processed.to_csv(RUN_DIR / "trpl_preprocessed.csv")
else:
    trpl_processed = trpl_processed_in

# remove non-finite / non-positive tau_diff, optional QFLS window, sort by QFLS
trpl = data.clean_trpl(trpl_processed, qfls_range=tr_cfg.get("qfls_window"))
trpl.to_csv(RUN_DIR / "trpl_used.csv", index=False)
print(f"tr-PL used for the analysis: {len(trpl)} points, QFLS = {trpl.qfls.min():.4f} … {trpl.qfls.max():.4f} eV")

### 3b. ss-PL: units and comparison mode
No conversion PLQY → QFLS is needed: the likelihood compares in the measured space
(`qfls_vs_suns`: QFLS [eV] vs intensity; `plqy_vs_suns` / `plqy_vs_qfls`: $\log_{10}$PLQY).
For intensities in suns the volumetric generation rate at 1 sun is computed from AM1.5G
(step-function absorption above $E_g$, absorptance and film thickness from the config).

In [ ]:
if HAVE_SSPL:
    SSPL_MODE = sspl_tf.canonical_mode(ss_cfg["mode"])
    # 1-sun generation: from the measured Voc,rad (sspl.voc_rad_V, k_rad fixed) or the step-absorption estimate.
    # Measured QFLS = Voc,rad + kT ln PLQY; model QFLS = kT ln(np/n_i^2) = Voc,rad,model + kT ln PLQY_model with
    # Voc,rad,model = kT ln(1 + G/(k_rad n_i^2)) -> same calibration when G reproduces the measured Voc,rad.
    G_PER_SUN, VOC_RAD_CHECK = (data.sspl_g_per_sun(cfg, art.constants) if SSPL_MODE.endswith("_suns") else (None, None))
    sspl_x, sspl_y = data.sspl_xy(sspl_df, SSPL_MODE)
    if G_PER_SUN:
        print(f"G(1 sun) = {G_PER_SUN:.4e} cm^-3 s^-1 ({VOC_RAD_CHECK['source']}; absorptance estimate "
              f"{VOC_RAD_CHECK['G_absorptance']:.4e} with absorptance {ss_cfg['absorptance']}, d = {d_cm*1e7:.0f} nm)")
        print("Voc,rad check:", json.dumps({k: (round(v, 5) if isinstance(v, float) else v)
                                            for k, v in VOC_RAD_CHECK.items()}))
    print(f"ss-PL comparison mode: {SSPL_MODE}, {len(sspl_x)} points")
    pd.DataFrame({"x": sspl_x, "y": sspl_y}).to_csv(RUN_DIR / f"sspl_used_{SSPL_MODE}.csv", index=False)

## 4. Likelihood — the single objective of the whole workflow
$$\log L_d = -\frac{\mathrm{MSE}_d}{2\sigma_d^2} - \log(\sigma_d\sqrt{2\pi}),\qquad
\log L = \sum_d \log L_d \quad (d = \text{tr-PL curve, ss-PL data})$$
(the MSE form of the original submission: each curve counts as one effective data point, which makes the
intervals conservative).  The same `LogPosterior` object is used by the optimiser, the samplers and the profile
likelihood.  tr-PL points outside the NN QFLS axis cannot be predicted and are excluded here.

The benchmark below measures likelihood + gradient evaluations per batch size (`timeit`; the first call per
batch size includes the XLA compilation and is reported separately).

In [ ]:
# sigmas of the terms during the fits of section 6 (6b sets the final sigmas with set_sigma).  sigma_rule
# 'model_plus_nn' (20/10/2026): the final sigmas need no fit (no MSE_min term), so they are used from the start —
# sigma_model (trpl.sigma) may then be 0 (synthetic data: sigma = sigma_nn).
FIT_SIGMA = {"trpl": float(tr_cfg["sigma"]), "sspl": float(ss_cfg["sigma"]) if HAVE_SSPL else None}
if cfg.get("sigma_rule") == "model_plus_nn":
    from inference.cylinder import nn_test_mse_log10
    # trpl.sigma_nn (20/10/2026): sigma_nn fixed in the config (e.g. the NN error at the best fit) instead of the
    # whole-curve test error of the network
    _sigma_nn = float(tr_cfg["sigma_nn"]) if tr_cfg.get("sigma_nn") is not None else float(np.sqrt(nn_test_mse_log10(nn_dir)))
    FIT_SIGMA["trpl"] = float(np.hypot(tr_cfg["sigma"], _sigma_nn))
    if HAVE_SSPL and ss_cfg.get("sigma_from_nn", False):
        FIT_SIGMA["sspl"] = _sigma_nn

# tr-PL: NN surrogate for the measured curve and the observed log10 tau_diff
sur = surrogate.make_trpl_surrogate(art, npulse, Eg, trpl.qfls.values, interp_space=tr_cfg["interp_space"])
term_trpl = likelihood.make_trpl_term(sur, trpl.qfls.values, trpl.tau_diff.values, sigma=FIT_SIGMA["trpl"])
n_out = int((~sur.mask_used).sum())
print(f"tr-PL points in the likelihood: {sur.n_points}" +
      (f"  ({n_out} points outside the NN QFLS axis [{sur.qfls_axis[0]:.4f}, {sur.qfls_axis[-1]:.4f}] eV excluded)" if n_out else ""))

# ss-PL: TF forward model (float64) + observation in the chosen mode
term_sspl = None
if HAVE_SSPL:
    lo, hi, n = ss_cfg["n_sweep"]
    sspl_model = sspl_tf.SsplModel(Eg, N_TRAPS, art.constants["Nc"], art.constants["Nv"], T=T,
                                   n_sweep=(lo, hi, n))
    sspl_obs = sspl_tf.SsplObservation(sspl_model, SSPL_MODE, sspl_x, sspl_y, G_per_sun=G_PER_SUN)
    # sspl.offset_sigma_eV (24/10/2026): constant ss-PL offset with a Gaussian prior, integrated out (likelihood.SsplTerm)
    term_sspl = likelihood.SsplTerm(sspl_obs, sigma=FIT_SIGMA["sspl"], offset_sigma=ss_cfg.get("offset_sigma_eV"))

# One LogPosterior per mode; they share the surrogate and the ss-PL model
LP = {"trpl": likelihood.LogPosterior(prior, [term_trpl], None, "trpl", batch_size=cfg["hardware"]["batch_size"])}
if HAVE_SSPL:
    LP["sspl"]  = likelihood.LogPosterior(prior, [], term_sspl, "sspl", batch_size=cfg["hardware"]["batch_size"])
    LP["joint"] = likelihood.LogPosterior(prior, [term_trpl], term_sspl, "joint", batch_size=cfg["hardware"]["batch_size"])
log_post = LP[MODE]
print(json.dumps(log_post.describe(), indent=1))

In [ ]:
# Throughput of log-likelihood + gradient (unit test 5 / timing table); skipped with NO_TIMING
bench = None
if not NO_TIMING:
    bench = benchmark_log_post(log_post, batch_sizes=(1, 16, 64, 256, 1024, 4096), repeats=5)
    bench.to_csv(RUN_DIR / "benchmark_loglik_grad.csv", index=False)
    fig, ax = plt.subplots(figsize=(5, 3.5))
    ax.loglog(bench.batch_size, bench.time_per_call_ms, "o-")
    ax.set_xlabel("batch size"); ax.set_ylabel("time per call [ms]"); ax.grid(alpha=0.3)
    ax.set_title(f"log L + gradient ({MODE})"); report.save_figure(fig, FIG_DIR / "4_benchmark.png"); plt.show()
    bench

## 5. Start-point estimation (physics-informed)
Curvature analysis of $\tau_\mathrm{diff}$(QFLS) with the legacy estimator (`start_point_estimation_module`),
one call per trap with the settings of the config.  The vector is projected into the prior support
(traps relabelled so that $E_{t1}<E_{t2}$).  Timed with `timeit` (`TIMING_REPEATS` repeats).

In [ ]:
if cfg["start_point"].get("explicit"):     # start values given explicitly (e.g. the published ones of the paper)
    sp_traps = cfg["start_point"]["explicit"]
    start, t_sp = timeit_call(start_points.explicit_start_point, prior, sp_traps,
                              repeats=TIMING_REPEATS, warmup=WARMUP, label="start-point (explicit, no estimation)")
else:                                       # curvature analysis of tau_diff(QFLS) (legacy estimator)
    sp_traps = cfg["start_point"]["traps"]
    start, t_sp = timeit_call(start_points.estimate_start_point, trpl, Eg, prior, sp_traps,
                              repeats=TIMING_REPEATS, warmup=WARMUP, label="start-point estimation")
log_timing(t_sp)
theta_start = start["theta"]
display(start["table"])

# quality of the start point under the likelihood
mse_start = {m: float(v[0]) for m, v in LP["joint" if HAVE_SSPL else "trpl"].np_mse_terms(theta_start[None]).items()}
print("MSE at the start point:", mse_start)
report.save_json({"theta": theta_start, "names": prior.names, "theta_raw": start["theta_raw"],
                  "physical": prior.to_physical(theta_start).iloc[0].to_dict(),
                  "estimator_table": start["table"], "mse": mse_start, "settings": sp_traps},
                 RUN_DIR / "start_point.json")

# plot: data vs NN prediction at the start point (and ss-PL if available)
fig, ax = report.plot_trpl_fit(LP["trpl"], theta_start, title="start point (NN)")
for m in start["masks"]:
    ax[0].semilogy(trpl.qfls[m], trpl.tau_diff[m], "s", mfc="none", ms=6, color="C1")
report.save_figure(fig, FIG_DIR / "5_start_point_trpl.png"); plt.show()
if HAVE_SSPL:
    fig, _ = report.plot_sspl_fit(LP["joint"], theta_start, title="start point (ss-PL model)")
    report.save_figure(fig, FIG_DIR / "5_start_point_sspl.png"); plt.show()

## 6. Fitting — maximum of the same likelihood, refined with the full rate equations
1. **CMA-ES** (gradient-free, as in the original submission) on the NN likelihood, from the physics-informed start
   and from `n_extra_sobol_starts` space-filling Sobol starts; all instances run in lockstep (each generation of all
   of them is *one* batched NN call).  **L-BFGS** (autodiff) polishes the best NN solutions.
2. **`scipy.optimize.curve_fit` with the full ODE** (option `RUN_ODE_REFINE`): CMA-ES can stall in flat valleys
   and inherits the surrogate error, so every CMA-ES result is refined with the rate equations (the solver that
   generated the training data).  Residuals are weighted with $\sigma_d\sqrt{N_d}$, i.e. curve_fit maximises the
   *same* likelihood, only with the ODE.  The starts run in parallel on CPU cores.
3. **$\sigma_\mathrm{eff}$** (legacy convention): $\sigma_{\mathrm{eff},d}^2 = \sigma_d^2 + \mathrm{MSE}_{\min,d}$ with the NN best-fit MSE of data set $d$
   fitted *alone* (consistent with the NN likelihood that is sampled in Section 7).
4. Plots: convergence of every start with the **switch from CMA-ES to curve_fit marked**, and a corner plot of the
   optimiser **trajectories** (best point per generation, one curve per start).
5. Exports: best-fit parameters as a human-readable `.dat`, the ODE simulation of the best fit and the NN prediction
   at the same point as CSV.

The final best fit (`theta_best`) is the best ODE-refined solution; `theta_best_nn` is the maximum of the NN likelihood.
Every step is timed with `timeit` (after XLA compilation) and its evaluations are counted.

In [ ]:
fit_cfg = cfg["fit"]
FITS = {}        # mode -> {"nn": FitResult (CMA-ES + L-BFGS), "ode": list of OdeFitResult or None}
BEST = {}        # mode -> final best theta (ODE-refined if available)
BEST_NN = {}     # mode -> maximum of the NN likelihood
PHYS_FIT = {}    # mode -> fit that started from the physics-informed start point

def run_fit(mode, theta_starts, labels, tag, refine_ode=True):
    """CMA-ES (+ L-BFGS) on LP[mode], then curve_fit with the ODE from every CMA-ES result."""
    lp = LP[mode]
    # compile the XLA programs first so that timeit measures the fit only
    t_comp = lp.precompile(max_rows=len(theta_starts) * 16)
    lp.tf_neg_log_lik_and_grad_u(tf.zeros([max(1, fit_cfg["n_polish"]), D], tf.float64))
    res, t = timeit_call(optimize.fit, lp, theta_starts, labels=labels, n_polish=fit_cfg["n_polish"],
                         cmaes_kwargs=dict(sigma0=fit_cfg["sigma0"], max_evals=fit_cfg["max_evals"],
                                           tolfun=fit_cfg["tolfun"], tolx=fit_cfg["tolx"], verbose=False),
                         repeats=TIMING_REPEATS, warmup=WARMUP, label=f"fit {tag}: CMA-ES + L-BFGS (NN)")
    log_timing(t, n_evals=res.n_evals_cmaes + res.n_evals_lbfgs, n_grad_evals=res.n_grad_evals_lbfgs,
               note=f"{len(theta_starts)} CMA-ES starts + {fit_cfg['n_polish']} L-BFGS; XLA compile {t_comp:.1f} s excluded")
    ode_res = None
    if refine_ode and RUN_ODE_REFINE and mode != "sspl":
        prob = refine.make_problem(lp, art, npulse, T)
        ode_res, t2 = timeit_call(refine.refine_many, prob, np.stack([r.theta for r in res.cmaes]),
                                  labels=[r.label for r in res.cmaes], max_nfev=fit_cfg["ode_max_nfev"],
                                  n_jobs=fit_cfg["n_jobs"], repeats=TIMING_REPEATS, warmup=WARMUP,
                                  label=f"fit {tag}: curve_fit with the ODE")
        log_timing(t2, n_evals=int(sum(r.n_ode_solves for r in ode_res)),
                   note=f"ODE solves; {len(ode_res)} starts in parallel, {fit_cfg['n_jobs']} worker processes on {os.cpu_count()} logical CPUs")
    FITS[mode] = {"nn": res, "ode": ode_res}
    BEST_NN[mode] = res.best.theta
    i_phys = [r.label for r in res.cmaes].index("physics") if "physics" in [r.label for r in res.cmaes] else 0
    if ode_res:
        BEST[mode] = min(ode_res, key=lambda r: r.neg_log_lik).theta
        PHYS_FIT[mode] = ode_res[i_phys].theta
    else:
        BEST[mode] = res.best.theta
        PHYS_FIT[mode] = res.cmaes[i_phys].theta
    # table: one row per start
    tab = res.table(prior)
    if ode_res:
        tab.insert(3, "ode_start_neg_log_lik", [r.start_neg_log_lik for r in ode_res])
        tab.insert(4, "ode_end_neg_log_lik", [r.neg_log_lik for r in ode_res])
        tab.insert(5, "ode_solves", [r.n_ode_solves for r in ode_res])
        for i, n in enumerate(prior.names):
            tab[f"ode_{n}"] = [r.theta[i] for r in ode_res]
    tab.to_csv(RUN_DIR / f"6_fit_runs_{mode}.csv", index=False)
    cols = ["label", "neg_log_lik", "n_evals"] + (["ode_start_neg_log_lik", "ode_end_neg_log_lik", "ode_solves"] if ode_res else []) + ["stop"]
    display(tab[cols].rename(columns={"neg_log_lik": "cmaes_neg_log_lik(NN)", "n_evals": "cmaes_evals"}))
    return res, ode_res

# 6a: tr-PL only (always: needed for sigma_eff of the tr-PL term)
sobol = prior.sobol(fit_cfg["n_extra_sobol_starts"], seed=1) if fit_cfg["n_extra_sobol_starts"] else np.zeros((0, D))
starts = np.vstack([theta_start, sobol])
labels = ["physics"] + [f"sobol_{i}" for i in range(len(sobol))]
fit_trpl = run_fit("trpl", starts, labels, "tr-PL only")

In [ ]:
# 6b: sigma_eff and (for MODE='joint') the joint fit
SIGMA = {"trpl_base": tr_cfg["sigma"], "sspl_base": ss_cfg["sigma"] if HAVE_SSPL else None,
         "trpl_fit": FIT_SIGMA["trpl"], "sspl_fit": FIT_SIGMA["sspl"]}   # *_fit: sigma of the section-6a fits (curve_fit cov)
mse_min_trpl = float(LP["trpl"].np_mse_terms(BEST_NN["trpl"][None])["trpl"][0])
SIGMA["mse_min_trpl"] = mse_min_trpl
if HAVE_SSPL:
    # ss-PL alone (exact physics model, cheap; no ODE refinement needed) -> MSE_min of the ss-PL term
    run_fit("sspl", np.vstack([BEST["trpl"], theta_start, sobol]), ["trpl_best", "physics"] + labels[1:],
            "ss-PL only", refine_ode=False)
    SIGMA["mse_min_sspl"] = float(LP["sspl"].np_mse_terms(BEST_NN["sspl"][None])["sspl"][0])

if cfg.get("sigma_rule") == "model_plus_nn":
    # sigma_trpl = sqrt(sigma_model^2 + sigma_nn^2): model error (trpl.sigma) + whole-curve test error of the network
    # (training_details.json test_mse_log10); no MSE_min term.  ss-PL: exact model -> sspl.sigma as given.
    from inference.cylinder import nn_test_mse_log10
    SIGMA["sigma_nn_trpl"] = (float(tr_cfg["sigma_nn"]) if tr_cfg.get("sigma_nn") is not None
                              else float(np.sqrt(nn_test_mse_log10(nn_dir))))
    SIGMA["sigma_nn_source"] = tr_cfg.get("sigma_nn_source", "config trpl.sigma_nn") if tr_cfg.get("sigma_nn") is not None \
        else "network test MSE (training_details.json test_mse_log10)"
    SIGMA["trpl"] = float(np.hypot(SIGMA["trpl_base"], SIGMA["sigma_nn_trpl"]))
    SIGMA["sspl"] = SIGMA["sspl_base"] if HAVE_SSPL else None
    SIGMA["rule"] = "trpl: sqrt(sigma_model^2 + sigma_nn^2); sspl: sigma_model"
    if HAVE_SSPL and ss_cfg.get("sigma_from_nn", False):
        # Robin, 19/10/2026 (synthetic 1-trap case): the ss-PL sigma is sigma_nn of the tr-PL surrogate as well
        # (ss-PL residual in log10 PLQY, i.e. decades like log10 tau_diff)
        SIGMA["sspl"] = SIGMA["sigma_nn_trpl"]
        SIGMA["rule"] = "trpl: sqrt(sigma_model^2 + sigma_nn^2); sspl: sigma_nn (sspl.sigma_from_nn)"
elif cfg["sigma_eff"]:
    SIGMA["trpl"] = float(np.sqrt(SIGMA["trpl_base"] ** 2 + mse_min_trpl))
    SIGMA["sspl"] = float(np.sqrt(SIGMA["sspl_base"] ** 2 + SIGMA["mse_min_sspl"])) if HAVE_SSPL else None
else:
    SIGMA["trpl"], SIGMA["sspl"] = SIGMA["trpl_base"], SIGMA["sspl_base"]
for lp in LP.values():                       # same sigmas in every LogPosterior
    lp.set_sigma(trpl=SIGMA["trpl"], sspl=SIGMA["sspl"])
print("sigmas:", json.dumps(SIGMA, indent=1))

if MODE == "joint":
    run_fit("joint", np.vstack([BEST["trpl"], theta_start, sobol]), ["trpl_best", "physics"] + labels[1:], "joint")

theta_best, theta_best_nn = BEST[MODE], BEST_NN[MODE]
ode_best = (min(FITS[MODE]["ode"], key=lambda r: r.neg_log_lik) if FITS[MODE]["ode"] else None)
prob_mode = refine.make_problem(log_post, art, npulse, T) if MODE != "sspl" else None

# summary: final best fit (ODE-refined) and the NN maximum, evaluated with NN and ODE
best_table = pd.concat([prior.to_physical(theta_best).T.rename(columns={0: "best fit (ODE-refined)"}),
                        prior.to_physical(theta_best_nn).T.rename(columns={0: "NN maximum"})], axis=1)
for name, th in (("best fit (ODE-refined)", theta_best), ("NN maximum", theta_best_nn)):
    best_table.loc["-log L (exact ss-PL model)" if MODE == "sspl" else "-log L (NN)", name] = -log_post.np_log_lik_theta(th[None])[0]
    if prob_mode is not None:             # mode sspl: no surrogate, nothing to refine with the ODE
        best_table.loc["-log L (ODE)", name] = prob_mode.neg_log_lik(th)
    for m, v in LP["joint" if HAVE_SSPL else "trpl"].np_mse_terms(th[None]).items():
        best_table.loc[f"RMSE {m} (NN)", name] = np.sqrt(v[0])
best_table

In [ ]:
# 6c: convergence of every start (switch CMA-ES -> curve_fit marked) and trajectory corner plot
fig, _ = report.plot_fit_histories(FITS[MODE]["nn"].cmaes, FITS[MODE]["ode"],
                                   title=f"fit convergence ({MODE}): CMA-ES (NN) then curve_fit (ODE)")
report.save_figure(fig, FIG_DIR / "6_fit_convergence.png"); plt.show()
# data of the convergence plot (Robin, 20/10/2026), long format: per start and stage one row per CMA-ES generation
# (cumulative NN evaluations) / curve_fit residual evaluation (cumulative ODE solves), best -log L so far
_rows = []
for _stage, _res in (("cmaes_nn", FITS[MODE]["nn"].cmaes), ("curve_fit_ode", FITS[MODE]["ode"] or [])):
    for _r in _res:
        if _r.history_evals is None:
            continue
        _rows += [{"stage": _stage, "label": _r.label, "step": _i, "evaluations": int(_e), "best_neg_log_lik": float(_f)}
                  for _i, (_e, _f) in enumerate(zip(_r.history_evals, _r.history_best))]
pd.DataFrame(_rows).to_csv(RUN_DIR / f"6_fit_convergence_{MODE}.csv", index=False)
print(f"convergence data: {RUN_DIR / f'6_fit_convergence_{MODE}.csv'} ({len(_rows)} rows)")
truth_theta = None
if cfg.get("true_params"):
    tp = cfg["true_params"]
    truth_theta = prior.from_physical(tp["Et_eV"], tp["Nt_cm3"], tp["tau_n_s"], tp["tau_p_s"], tp["krad_cm3s"])
truth_disp = prior.to_display(truth_theta)[0] if truth_theta is not None else None
fig = report.plot_trajectory_corner(prior, FITS[MODE]["nn"].cmaes, FITS[MODE]["ode"], truth=truth_disp,
                                    title=f"optimiser trajectories ({MODE})")
report.save_figure(fig, FIG_DIR / "6_fit_trajectories_corner.png"); plt.show()

### 6d. Best fit vs data and check with the rate equations
NN prediction and ODE simulation at the final best fit.  The cost of one ODE solve is measured with `timeit`
(at least 20 repeats; its mean ± SD is the basis of the ODE estimates in the timing table).

In [ ]:
ode_curve, ODE_CHECK = None, {}
if RUN_ODE_CHECK:
    ode.simulate_trpl(theta_best, prior, npulse, art.constants, T)        # numba compilation
    ode_curve, t_ode = timeit_call(ode.simulate_trpl, theta_best, prior, npulse, art.constants, T,
                                   repeats=1 if NO_TIMING else max(20, TIMING_REPEATS), warmup=WARMUP, label="one ODE solve (best fit)")
    log_timing(t_ode, n_evals=1)
    for name, th in (("best_fit", theta_best), ("nn_maximum", theta_best_nn)):
        y_ode = ode.ode_log10_tau(th, prior, npulse, art.constants, sur.qfls_used, T)
        ok = np.isfinite(y_ode)
        y_nn = LP["trpl"].predict_trpl(th)["log10_tau_used"][0]
        ODE_CHECK[name] = {"mse_trpl_nn": float(np.mean((y_nn - term_trpl.y_obs) ** 2)),
                           "mse_trpl_ode": float(np.mean((y_ode[ok] - term_trpl.y_obs[ok]) ** 2)),
                           "rms_nn_minus_ode": float(np.sqrt(np.mean((y_nn[ok] - y_ode[ok]) ** 2))),
                           "max_abs_nn_minus_ode": float(np.max(np.abs(y_nn[ok] - y_ode[ok]))),
                           "fraction_of_points_covered_by_ode": float(ok.mean())}
    ODE_CHECK.update(ode_seconds_per_solve=t_ode.mean_s, ode_seconds_per_solve_sd=t_ode.std_s,
                     ode_solve_repeats=len(t_ode.times_s))
    print(json.dumps(ODE_CHECK, indent=1))

fig, _ = report.plot_trpl_fit(LP["trpl"], theta_best, ode_curve=ode_curve,
                              title=f"best fit ({MODE}) — tr-PL: NN and ODE at the ODE-refined fit"
                              + (" (prediction, not fitted)" if MODE == "sspl" else ""))
report.save_figure(fig, FIG_DIR / "6_best_fit_trpl.png"); plt.show()
if HAVE_SSPL:
    _d = float(LP["joint"].sspl_offset(theta_best)[0])     # marginalised ss-PL offset (0 without sspl.offset_sigma_eV)
    fig, _ = report.plot_sspl_fit(LP["joint"], theta_best, title=f"best fit ({MODE}) — ss-PL"
                                  + ("" if MODE in ("joint", "sspl") else " (prediction, not fitted)")
                                  + (f"; offset delta_hat = {_d * 1e3:+.1f} meV (model - data)" if _d else ""))
    report.save_figure(fig, FIG_DIR / "6_best_fit_sspl.png"); plt.show()

### 6e. Export of the best fit
* `6_best_fit_parameters.dat` — final (ODE-refined) best fit, human readable, with 1σ from the curve_fit covariance
* `6_best_fit_parameters_nn.dat` — maximum of the NN likelihood
* `6_best_fit_ode_simulation.csv` — full ODE output at the best fit (time, PL, QFLS, τ_diff, n, p, trap occupations, capture/emission rates)
* `6_best_fit_nn_simulation.csv` — NN prediction at the same point (on the NN QFLS axis and at the data points)
* `6_best_fit.json` — everything in machine-readable form

In [ ]:
extra = {"run": cfg["run_name"], "mode": MODE, "network": nn_dir.name,
         "-log L (ODE)": f"{prob_mode.neg_log_lik(theta_best):.6g}" if prob_mode is not None else "— (mode sspl)",
         "-log L (NN)" if MODE != "sspl" else "-log L (exact ss-PL model)": f"{-log_post.np_log_lik_theta(theta_best[None])[0]:.6g}",
         "sigma_trpl": SIGMA["trpl"], "sigma_sspl": SIGMA["sspl"],
         "fixed parameters": cfg["prior"].get("fixed", {}) or "none",
         "method": "CMA-ES (NN) -> curve_fit (ODE)" if ode_best is not None else
                   ("CMA-ES + L-BFGS (exact ss-PL model)" if MODE == "sspl" else "CMA-ES + L-BFGS (NN)")}
report.write_parameters_dat(RUN_DIR / "6_best_fit_parameters.dat", prior, theta_best,
                            title=f"best fit (ODE-refined) — {cfg['run_name']} ({MODE})", extra=extra,
                            cov_theta=ode_best.cov_theta if ode_best is not None else None)
report.write_parameters_dat(RUN_DIR / "6_best_fit_parameters_nn.dat", prior, theta_best_nn,
                            title=f"maximum of the NN likelihood — {cfg['run_name']} ({MODE})")
ode_sim = ode.simulate_trpl_full(theta_best, prior, npulse, art.constants, T)
ode_sim.to_csv(RUN_DIR / "6_best_fit_ode_simulation.csv", index=False, float_format="%.8g")
p_nn = LP["trpl"].predict_trpl(theta_best)
# qfls_eV: calibration of the data (high-injection formula); axis-head networks also give the physical QFLS on the
# network's q_max(theta_best) (qfls_physical_eV), the ODE file has both (qfls_eV physical, qfls_nominal_eV)
nn_sim = {"qfls_eV": p_nn["qfls_axis"], "log10_tau_diff_nn": p_nn["log10_tau_axis"][0],
          "tau_diff_nn_s": 10 ** p_nn["log10_tau_axis"][0]}
if "qfls_axis_physical" in p_nn:
    nn_sim.update({"qfls_physical_eV": p_nn["qfls_axis_physical"][0],
                   "tau_diff_nn_on_physical_axis_s": 10 ** p_nn["log10_tau_own_axis"][0]})
pd.DataFrame(nn_sim).to_csv(RUN_DIR / "6_best_fit_nn_simulation.csv", index=False, float_format="%.8g")
at_data = {"qfls_eV": p_nn["qfls_used"], "tau_diff_data_s": 10 ** p_nn["log10_tau_obs"],
           "tau_diff_nn_s": 10 ** p_nn["log10_tau_used"][0],
           "tau_diff_ode_s": 10 ** ode.ode_log10_tau(theta_best, prior, npulse, art.constants, p_nn["qfls_used"], T)}
if "qfls_used_physical" in p_nn:
    at_data["qfls_physical_eV"] = p_nn["qfls_used_physical"][0]
pd.DataFrame(at_data).to_csv(RUN_DIR / "6_best_fit_at_data_points.csv", index=False, float_format="%.8g")
if HAVE_SSPL:
    c = LP["joint"].predict_sspl(theta_best)["curve"]
    pd.DataFrame({"x_" + SSPL_MODE.split("_vs_")[1]: c["x"][0], "y_" + SSPL_MODE.split("_vs_")[0]: c["y"][0],
                  "n": c["n"][0], "p": c["p"][0], "G": c["G"][0],
                  **{k: c[k][0] for k in sorted(c) if k.startswith("nt_")}}).to_csv(RUN_DIR / "6_best_fit_sspl_curve.csv", index=False)
report.save_json({"mode": MODE, "names": prior.names, "theta_best": theta_best, "theta_best_nn": theta_best_nn,
                  "physical": prior.to_physical(theta_best).iloc[0].to_dict(),
                  "physical_nn_maximum": prior.to_physical(theta_best_nn).iloc[0].to_dict(),
                  "cov_theta_curve_fit": ode_best.cov_theta if ode_best is not None else None,
                  "sigmas": SIGMA, "fixed": cfg["prior"].get("fixed", {}), "voc_rad_check": globals().get("VOC_RAD_CHECK"),
                  "best_per_mode": {m: {"theta": t, "physical": prior.to_physical(t).iloc[0].to_dict()} for m, t in BEST.items()},
                  "evaluations": {m: {"cmaes": f["nn"].n_evals_cmaes, "lbfgs": f["nn"].n_evals_lbfgs,
                                      "ode_solves": int(sum(r.n_ode_solves for r in f["ode"])) if f["ode"] else 0}
                                  for m, f in FITS.items()},
                  "ode_check": ODE_CHECK,
                  "sspl_offset": None if not (HAVE_SSPL and ss_cfg.get("offset_sigma_eV")) else
                      {"sigma_delta_eV": ss_cfg["offset_sigma_eV"], "note": "delta_hat = most probable offset, model - data",
                       "delta_hat_eV": float(LP["joint"].sspl_offset(theta_best)[0]),
                       "delta_hat_eV_per_mode": {m: float(LP["joint"].sspl_offset(t)[0]) for m, t in BEST.items()}}},
                 RUN_DIR / "6_best_fit.json")
print("exported best fit to", RUN_DIR)

### 6f. Same CMA-ES fit with the rate equations instead of the NN (optional, slow)
Identical start, step size and tolerances; tr-PL likelihood evaluated with the ODE (populations solved in
parallel on CPU).  Gives the explicit "optimisation with the ODE" entry of the timing table (R1 comment 7, R2 comment 2).

In [ ]:
if RUN_ODE_FIT:
    ode_obj = ode.OdeTrplObjective(prior, npulse, art.constants, sur.qfls_used, term_trpl.y_obs,
                                   sigma=SIGMA["trpl"], n_jobs=cfg["ode"]["n_jobs"], T=T)
    ode_obj.np_neg_log_lik_u(np.zeros((2, D)))            # start the worker pool (numba compile)
    def ode_fit_once():
        ode_obj.counter.reset()            # evaluations of the last repeat only
        return optimize.cmaes_multistart(ode_obj, theta_start[None], sigma0=fit_cfg["sigma0"],
                                         max_evals=fit_cfg["max_evals"], tolfun=fit_cfg["tolfun"],
                                         tolx=fit_cfg["tolx"], labels=["physics"], verbose=False)
    optimize.cmaes_multistart(ode_obj, theta_start[None], sigma0=fit_cfg["sigma0"], max_evals=100,   # untimed warm-up
                              tolfun=fit_cfg["tolfun"], tolx=fit_cfg["tolx"], labels=["physics"], verbose=False)
    ode_fit, t_odefit = timeit_call(ode_fit_once, repeats=TIMING_REPEATS_LONG, label="fit tr-PL with ODE (CMA-ES)")
    log_timing(t_odefit, n_evals=ode_obj.counter.snapshot()["n_points"],
               note=f"{cfg['ode']['n_jobs']} worker processes on {os.cpu_count()} logical CPUs")
    nn_phys = FITS["trpl"]["nn"].cmaes[0]
    print(f"ODE fit: -log L = {ode_fit[0].neg_log_lik:.4f} after {ode_fit[0].n_evals} evaluations; "
          f"NN fit (same start): {nn_phys.neg_log_lik:.4f} after {nn_phys.n_evals} evaluations")
    report.save_json({"ode_fit_theta": ode_fit[0].theta, "ode_fit_neg_log_lik": ode_fit[0].neg_log_lik,
                      "ode_fit_n_evals": ode_fit[0].n_evals, "time": t_odefit.as_dict()}, RUN_DIR / "6f_ode_fit.json")

## 7. Inference — full-dimensional posterior sampling
* **NUTS** (TensorFlow Probability) on $\log p(u) = \log L + \log\pi + \log|J|$ in unconstrained coordinates;
  all chains are one batch.  Warm-up in three stages with dense-metric and step-size adaptation; chains that
  are stuck in regions of negligible posterior mass after warm-up stages 1/2 are moved onto good chains
  (reported).  Chains start over-dispersed over the prior (`init = 'dispersed'`) or at the best fit.
* **emcee** (affine-invariant ensemble, stretch move) samples the same density as a cross-check; walkers stuck
  on plateaus of negligible mass after burn-in are moved onto good walkers before the production run (reported).
* Diagnostics: rank-normalised $\hat R$ (< 1.01), bulk/tail ESS, divergences; emcee: integrated autocorrelation time.
* Credible intervals: equal-tailed 68 % / 95 %; parameters whose marginal reaches a prior bound are flagged
  ("< x", "> x").  Derived Eq. 10/11 combinations are reported as well.

In [ ]:
nuts_cfg = cfg["nuts"]
rng = np.random.default_rng(nuts_cfg["seed"])
u_best = prior.inverse(theta_best_nn[None])[0]   # NN posterior: start around the NN maximum
nuts_res, diag_nuts, info = None, None, None
if RUN_NUTS:
    init_u = (samplers.init_dispersed(D, nuts_cfg["n_chains"], rng) if nuts_cfg["init"] == "dispersed"
              else samplers.init_around(u_best, nuts_cfg["n_chains"], rng))

    # XLA compilation in a short untimed run; the timed repeats reuse the compiled sampler
    # (identical runs: same start points and seed), so every repeat is compile-free.
    (_, nuts_sampler), t_nuts_compile = timeit_call(
        samplers.run_nuts, log_post, init_u, n_warmup=WARMUP, n_draws=1, metric=nuts_cfg["metric"],
        target_accept=nuts_cfg["target_accept"], max_tree_depth=nuts_cfg["max_tree_depth"],
        seed=nuts_cfg["seed"], verbose=False, label="NUTS compile (short untimed run)")
    (nuts_res, nuts_sampler), t_nuts = timeit_call(
        samplers.run_nuts, log_post, init_u, n_warmup=nuts_cfg["n_warmup"], n_draws=nuts_cfg["n_draws"],
        metric=nuts_cfg["metric"], target_accept=nuts_cfg["target_accept"],
        max_tree_depth=nuts_cfg["max_tree_depth"], seed=nuts_cfg["seed"], sampler=nuts_sampler,
        repeats=TIMING_REPEATS_LONG, label="NUTS (warm-up + sampling)")
    info = nuts_res.info
    log_timing(t_nuts, n_evals=info["evals_total_points"], n_grad_evals=info["evals_total_points"],
               note=f"{info['n_chains']} chains x ({info['n_warmup']} + {info['n_draws']}); compiled in a short untimed run ({t_nuts_compile.best_s:.0f} s); "
                    f"leapfrogs used in sampling: {info['leapfrogs_sampling']}")
    diag_nuts = samplers.diagnostics(nuts_res, prior)
    print(f"divergences: {info['n_divergent']} | max R-hat: {diag_nuts['r_hat'].max():.4f} | "
          f"min ESS bulk/tail: {diag_nuts['ess_bulk'].min():.0f}/{diag_nuts['ess_tail'].min():.0f} | "
          f"chains reset in warm-up: {info['chains_reset']}")
    display(diag_nuts)
else:
    print("7a skipped (RUN_NUTS = False)")


In [ ]:
# 7b: emcee cross-check (optional)
emcee_res, diag_emcee = None, None
if RUN_EMCEE:
    em = cfg["emcee"]
    init_w = samplers.init_dispersed(D, em["n_walkers"], np.random.default_rng(em["seed"]))
    log_post.precompile(max_rows=em["n_walkers"], which=("lp",))
    emcee_res, t_em = timeit_call(samplers.run_emcee, log_post, init_w, n_steps=em["n_steps"],
                                  burn_frac=em["burn_frac"], moves=em["moves"], seed=em["seed"],
                                  repeats=TIMING_REPEATS_LONG, label="emcee")
    log_timing(t_em, n_evals=emcee_res.info["evals_total_points"], note=f"{em['n_walkers']} walkers x {em['n_steps']} steps")
    diag_emcee = samplers.diagnostics(emcee_res, prior)
    print(f"emcee: acceptance {emcee_res.info['acceptance_fraction']:.3f}, tau_int = "
          f"{np.round(emcee_res.info['tau_int'], 1)}, (production steps)/max tau = {emcee_res.info['steps_over_tau']:.1f} (want >= 50), "
          f"walkers reset after burn-in: {emcee_res.info['walkers_reset']}")
    display(diag_emcee)

In [ ]:
# 7c: credible intervals, prior-limited flags, derived quantities, NUTS vs emcee
keys = prior.display_keys
samples_nuts = prior.to_display(nuts_res.flat_theta()) if nuts_res is not None else None
truth_theta = None
if cfg.get("true_params"):
    tp = cfg["true_params"]
    truth_theta = prior.from_physical(tp["Et_eV"], tp["Nt_cm3"], tp["tau_n_s"], tp["tau_p_s"], tp["krad_cm3s"])
truth_disp = prior.to_display(truth_theta)[0] if truth_theta is not None else None
best_disp = prior.to_display(theta_best)[0]

if nuts_res is not None:
    ci = report.credible_intervals(samples_nuts, prior.display_bounds(), keys, truth=truth_disp, best=best_disp)
    ci_derived = report.derived_intervals(nuts_res.flat_theta(), prior)
    display(ci)
    display(ci_derived)

    if emcee_res is not None:
        samples_emcee = prior.to_display(emcee_res.flat_theta())
        ci_em = report.credible_intervals(samples_emcee, prior.display_bounds(), keys)
        # agreement of the medians relative to the Monte-Carlo error of both samplers.
        # emcee walkers are not independent chains, so its effective sample size is
        # taken from the integrated autocorrelation time: n_walkers * (steps after burn-in) / tau_int
        ei = emcee_res.info
        ess_emcee = ei["n_walkers"] * (ei["n_steps"] - ei["burn"]) / np.asarray(ei["tau_int"])
        se = np.sqrt(diag_nuts["sd"].values ** 2 / diag_nuts["ess_bulk"].values
                     + diag_emcee["sd"].values ** 2 / ess_emcee)
        cmp = pd.DataFrame({"median_nuts": ci["median"], "median_emcee": ci_em["median"],
                            "z_median": (ci["median"] - ci_em["median"]) / se,
                            "ess_nuts": diag_nuts["ess_bulk"].values, "ess_emcee_tau": ess_emcee,
                            "width68_nuts": ci["width68"], "width68_emcee": ci_em["width68"],
                            "width95_nuts": ci["width95"], "width95_emcee": ci_em["width95"]})
        print("NUTS vs emcee (|z_median| < 3 means agreement within Monte-Carlo error):")
        display(cmp)

In [ ]:
if nuts_res is not None:
    # 7d: plots — trace, corner (NUTS + emcee, truth, best fit), posterior predictive, marginals
    fig = report.plot_trace(nuts_res, prior)
    report.save_figure(fig, FIG_DIR / "7_trace_nuts.png"); plt.show()

    fig = report.plot_corner(samples_nuts, prior, truth=truth_disp, best=best_disp,
                             overlay=samples_emcee if emcee_res is not None else None)
    report.save_figure(fig, FIG_DIR / "7_corner.png"); plt.show()

    ppc = nuts_res.flat_theta()[np.random.default_rng(3).choice(len(samples_nuts), size=300, replace=False)]
    fig, _ = report.plot_trpl_fit(LP["trpl"], theta_best, theta_samples=ppc, title=f"posterior predictive ({MODE}) — tr-PL"
                                  + (" (not fitted)" if MODE == "sspl" else ""))
    report.save_figure(fig, FIG_DIR / "7_posterior_predictive_trpl.png"); plt.show()
    if HAVE_SSPL:
        fig, _ = report.plot_sspl_fit(LP["joint"], theta_best, theta_samples=ppc,
                                      title=f"posterior predictive ({MODE}) — ss-PL" + ("" if MODE in ("joint", "sspl") else " (not fitted)"))
        report.save_figure(fig, FIG_DIR / "7_posterior_predictive_sspl.png"); plt.show()
    if emcee_res is not None:
        fig = report.plot_marginals_compare({"NUTS": samples_nuts, "emcee": samples_emcee}, prior, truth=truth_disp)
        report.save_figure(fig, FIG_DIR / "7_marginals_nuts_vs_emcee.png"); plt.show()

In [ ]:
if nuts_res is not None:
    # 7e: export
    report.export_samples(nuts_res, prior, RUN_DIR / "posterior_samples_nuts.csv")
    diag_nuts.to_csv(RUN_DIR / "diagnostics_nuts.csv")
    ci.to_csv(RUN_DIR / "credible_intervals.csv")
    ci_derived.to_csv(RUN_DIR / "credible_intervals_derived.csv")
    report.save_json({"nuts": nuts_res.info, "emcee": emcee_res.info if emcee_res is not None else None,
                      "sigmas": SIGMA, "likelihood": log_post.describe()}, RUN_DIR / "sampling_info.json")
    if emcee_res is not None:
        report.export_samples(emcee_res, prior, RUN_DIR / "posterior_samples_emcee.csv")
        diag_emcee.to_csv(RUN_DIR / "diagnostics_emcee.csv")
        cmp.to_csv(RUN_DIR / "nuts_vs_emcee.csv")
    try:   # full ArviZ object (needs h5netcdf or netCDF4)
        samplers.to_arviz(nuts_res, prior).to_netcdf(RUN_DIR / "posterior_nuts.nc")
    except Exception as e:
        print("NetCDF export skipped:", type(e).__name__, e)
    print("exported to", RUN_DIR)

### 7f. Full-dimensional grid search (comparison with NUTS and emcee)
The same likelihood on a Cartesian grid with `GRID_POINTS_PER_AXIS` values per free parameter, all parameters varying
at once (not the pairwise slices of the original submission).  `GRID_RANGE = 'window'`: centred on the physics-informed
fit (half-widths from the config, clipped to the prior box); `'prior'`: the full box.  Uniform prior → every grid point
has the same weight; points violating the ordering are skipped.  Masses are cell probabilities; intervals have the
resolution of the grid spacing.  For a fair comparison the NUTS/emcee samples are also restricted to the grid window.
Cost: $n^D$ evaluations (estimate printed first).

In [ ]:
GRID_RES, cmp_grid = None, None
if RUN_GRID:
    gc = cfg["grid"]
    axes = grid.grid_axes(prior, PHYS_FIT[MODE], GRID_POINTS_PER_AXIS, GRID_RANGE,
                          gc["halfwidth_log10"], gc["halfwidth_Et_eV"])
    n_grid = grid.grid_size(axes)
    t_pt = bench.time_per_point_us.min() * 1e-6          # value+gradient throughput: an upper bound
    est = n_grid * t_pt
    print(f"grid: {GRID_POINTS_PER_AXIS} points per free axis ({len(prior.free_idx)} free), {n_grid:,} points; "
          f"estimated time < {est:.0f} s" + (f" ({est / 3600:.1f} h)" if est > 3600 else ""))
    log_post.precompile(max_rows=cfg["hardware"]["batch_size"], which=("ll",))   # compile before timing
    GRID_RES, t_grid = timeit_call(grid.grid_posterior, log_post, axes, repeats=TIMING_REPEATS_LONG,
                                   label=f"grid ({GRID_POINTS_PER_AXIS}^{len(prior.free_idx)} points)")
    log_timing(t_grid, n_evals=GRID_RES.n_evaluated,
               note=f"{n_grid:,} grid points, {GRID_RES.n_evaluated:,} inside the prior support")
    ci_grid = grid.grid_intervals(GRID_RES, prior)

    # samples restricted to the grid window (all parameters at once)
    half = [0.5 * (a[1] - a[0]) if len(a) > 1 else 1e-6 for a in axes]
    def in_window(th):
        return np.all([(th[:, i] >= axes[i][0] - half[i]) & (th[:, i] <= axes[i][-1] + half[i])
                       for i in range(D)], axis=0)
    def quantile_row(disp_samples):
        q = np.percentile(disp_samples, [2.5, 16, 50, 84, 97.5], axis=0)
        return q
    sets = {"NUTS (all)": nuts_res.flat_theta()}
    sets["NUTS (in window)"] = sets["NUTS (all)"][in_window(sets["NUTS (all)"])]
    if emcee_res is not None:
        th_e = emcee_res.flat_theta()
        sets["emcee (in window)"] = th_e[in_window(th_e)]
    rows = []
    for i in prior.free_idx:
        k = keys[i]
        r = {"parameter": k, "grid_median": ci_grid.loc[k, "median"], "grid_lo68": ci_grid.loc[k, "lo68"],
             "grid_hi68": ci_grid.loc[k, "hi68"], "grid_lo95": ci_grid.loc[k, "lo95"], "grid_hi95": ci_grid.loc[k, "hi95"],
             "grid_edge_mass_low": ci_grid.loc[k, "edge_mass_low"], "grid_edge_mass_high": ci_grid.loc[k, "edge_mass_high"]}
        for name, th in sets.items():
            if len(th) == 0:
                continue
            q = quantile_row(prior.to_display(th))[:, i]
            r.update({f"{name}_median": q[2], f"{name}_lo68": q[1], f"{name}_hi68": q[3],
                      f"{name}_lo95": q[0], f"{name}_hi95": q[4]})
        rows.append(r)
    cmp_grid = pd.DataFrame(rows).set_index("parameter")
    frac_in = len(sets["NUTS (in window)"]) / len(sets["NUTS (all)"])
    print(f"fraction of the NUTS posterior inside the grid window: {frac_in:.3f}")
    display(cmp_grid.T)

    fig = grid.plot_grid_vs_samples(GRID_RES, prior, {k: prior.to_display(v) for k, v in sets.items()
                                                      if k != "NUTS (all)" and len(v)}, truth=truth_disp)
    report.save_figure(fig, FIG_DIR / "7f_grid_vs_samplers_marginals.png"); plt.show()
    fig = grid.plot_grid_corner(GRID_RES, prior, samples_disp=prior.to_display(sets["NUTS (in window)"]),
                                truth=truth_disp)
    report.save_figure(fig, FIG_DIR / "7f_grid_corner.png"); plt.show()
    cmp_grid.to_csv(RUN_DIR / "7f_grid_vs_samplers.csv")
    ci_grid.to_csv(RUN_DIR / "7f_grid_intervals.csv")
    pd.concat([pd.DataFrame({"parameter": keys[i], "theta": GRID_RES.axes[i], "mass": GRID_RES.marg1[i]})
               for i in range(D)]).to_csv(RUN_DIR / "7f_grid_marginals_1d.csv", index=False)
    report.save_json({"points_per_axis": GRID_POINTS_PER_AXIS, "range": GRID_RANGE, "center_theta": PHYS_FIT[MODE],
                      "axes": GRID_RES.axes, "n_points": GRID_RES.n_points, "n_evaluated": GRID_RES.n_evaluated,
                      "log_z": GRID_RES.log_z, "theta_max": GRID_RES.theta_max, "log_lik_max": GRID_RES.log_lik_max,
                      "time_s": GRID_RES.time_s, "fraction_nuts_in_window": frac_in}, RUN_DIR / "7f_grid_info.json")

### 7g. Does the surrogate change the posterior? ODE re-evaluation with PSIS (optional, T7)
Thinned posterior samples are re-evaluated with the rate equations; $\log w = \log L_\mathrm{ODE} - \log L_\mathrm{NN}$
(only the tr-PL term differs), Pareto-smoothed importance sampling.  $\hat k < 0.7$: the re-weighting is reliable;
compare the NN and the ODE-re-weighted marginals.

In [ ]:
RW = None
if RUN_ODE_REWEIGHT and MODE == "sspl":
    print("7g skipped: mode 'sspl' has no surrogate term (the ss-PL model is exact)")
if RUN_ODE_REWEIGHT and MODE != "sspl":
    t7 = cfg["t7_ode_reweight"]
    RW, t_rw = timeit_call(validate.ode_reweight, log_post, art, npulse, nuts_res.flat_theta(),
                           n_samples=t7["n_samples"], n_jobs=fit_cfg["n_jobs"], T=T,
                           label=f"T7: ODE re-evaluation of {t7['n_samples']} posterior samples")
    log_timing(t_rw, n_evals=RW["n_used"] + RW["n_dropped"], note=f"ODE solves, {fit_cfg['n_jobs']} worker processes on {os.cpu_count()} logical CPUs")
    tab_rw = validate.reweight_table(RW, prior)
    print(f"Pareto k-hat = {RW['k_hat']:.3f}  (< 0.7: reliable),  ESS = {RW['ess']:.0f} of {RW['n_used']} "
          f"(dropped: {RW['n_dropped']} samples whose ODE curve does not cover the data)")
    display(tab_rw)
    fig = validate.plot_reweighted(RW, prior)
    report.save_figure(fig, FIG_DIR / "7g_ode_reweighting.png"); plt.show()
    tab_rw.to_csv(RUN_DIR / "7g_ode_reweighting.csv")
    report.save_json({k: v for k, v in RW.items() if k not in ("theta", "log_w", "log_w_psis", "mse_nn", "mse_ode")},
                     RUN_DIR / "7g_ode_reweighting.json")
    pd.DataFrame(np.column_stack([RW["theta"], RW["mse_nn"], RW["mse_ode"], RW["log_w"], RW["log_w_psis"]]),
                 columns=prior.names + ["mse_nn", "mse_ode", "log_w", "log_w_psis"]).to_csv(
        RUN_DIR / "7g_ode_reweighting_samples.csv", index=False)

### 7h. Trap dynamics across the posterior (optional, T9 / R1 comment 2, two traps only)
Figure 8 of the paper interprets the dynamics of ONE best fit: the net electron capture of defect 2 changes sign
during the decay and defect 1 captures the emitted electrons.  Here the same quantities are computed with the ODE
for posterior samples (trap 1 = deeper trap) inside the QFLS window of the data.

In [ ]:
DYN = None
if RUN_TRAP_DYNAMICS and N_TRAPS >= 2:
    t9 = cfg["t9_trap_dynamics"]
    DYN, t_dyn = timeit_call(dynamics.trap_dynamics, nuts_res.flat_theta(), prior, npulse, art.constants,
                             (float(sur.qfls_used.min()), float(sur.qfls_used.max())),
                             n_samples=t9["n_samples"], rel_tol=t9["rel_tol"], n_jobs=fit_cfg["n_jobs"], T=T,
                             theta_best=theta_best, label=f"T9: trap dynamics of {t9['n_samples']} posterior samples")
    log_timing(t_dyn, n_evals=DYN["n_used"] + DYN["n_failed"], note="ODE solves")
    T9 = {"n_samples": DYN["n_used"], "fraction_sign_change_trap2": DYN["frac_sign_change_2"],
          "fraction_transfer_2_to_1": DYN["frac_transfer_2_to_1"], "fraction_both": DYN["frac_both"],
          "best_fit_sign_change_trap2": DYN["best"].get("sign_change_2") if DYN["best"] else None,
          "best_fit_transfer_2_to_1": DYN["best"].get("transfer_2_to_1") if DYN["best"] else None,
          "rel_tol": DYN["rel_tol"]}
    print(json.dumps(T9, indent=1))
    fig = dynamics.plot_trap_dynamics(DYN)
    report.save_figure(fig, FIG_DIR / "7h_trap_dynamics.png"); plt.show()
    DYN["table"].to_csv(RUN_DIR / "7h_trap_dynamics_samples.csv", index=False)
    report.save_json(T9, RUN_DIR / "7h_trap_dynamics_summary.json")

## 8. Sensitivity to the start values (optional, T2 / R1 comment 1, R1 comment 7)
**8a. Optimiser** — CMA-ES from `n_sobol_starts` Sobol points over the prior plus the physics-informed start
(identical settings), then curve_fit with the ODE from every CMA-ES result.  Reported: final $-\log L$ per start
(after CMA-ES and after the ODE refinement), fraction of starts within `tol` of the best, evaluations to convergence for
physics-informed vs random starts; convergence curves with the **switch to curve_fit marked**, and one corner plot with
the **trajectories of all starts** (best point per generation as a connected curve).

**8b. Chains** — two NUTS runs: group A over-dispersed over the prior, group B started at the best fit.
If both groups describe the same posterior, $\hat R$ over all chains of both groups is < 1.01 and the
quantiles agree within Monte-Carlo error.

In [ ]:
if RUN_SENSITIVITY:
    sc = cfg["sensitivity"]
    s_starts = np.vstack([theta_start, prior.sobol(sc["n_sobol_starts"], seed=11)])
    s_labels = ["physics"] + [f"sobol_{i}" for i in range(sc["n_sobol_starts"])]
    log_post.precompile(max_rows=len(s_starts) * 16)
    # 1) CMA-ES (NN) from all starts, in lockstep
    with count_evals(log_post.counter) as c_sens:
        sens, t_sens = timeit_call(optimize.cmaes_multistart, log_post, s_starts, sigma0=fit_cfg["sigma0"],
                                   max_evals=sc["max_evals"], tolfun=fit_cfg["tolfun"], tolx=fit_cfg["tolx"],
                                   labels=s_labels, verbose=False, label=f"CMA-ES from {len(s_starts)} starts")
    log_timing(t_sens, n_evals=c_sens.counts["n_points"])
    # 2) curve_fit with the ODE from every CMA-ES result (parallel on CPU cores)
    sens_ode = None
    if RUN_ODE_REFINE:
        sens_ode, t_so = timeit_call(refine.refine_many, prob_mode, np.stack([r.theta for r in sens]),
                                     labels=s_labels, max_nfev=fit_cfg["ode_max_nfev"], n_jobs=fit_cfg["n_jobs"],
                                     label=f"curve_fit (ODE) from {len(sens)} CMA-ES results")
        log_timing(t_so, n_evals=int(sum(r.n_ode_solves for r in sens_ode)), note="ODE solves")
    sens_tab = optimize.FitResult(best=min(sens, key=lambda r: r.neg_log_lik), cmaes=sens).table(prior)
    f_glob = min(sens_tab.neg_log_lik.min(), -log_post.np_log_lik_theta(theta_best_nn[None])[0])
    tol = sc["tol_neg_log_lik"]
    sens_tab["delta_neg_log_lik"] = sens_tab.neg_log_lik - f_glob
    sens_tab["evals_to_reach_tol"] = [r.evals_to_reach(f_glob + tol) for r in sens]
    phys, rand = sens_tab.iloc[0], sens_tab.iloc[1:]
    summary_T2 = {"n_random_starts": len(rand), "tol_neg_log_lik": tol,
                  "cmaes_fraction_random_within_tol": float((rand.delta_neg_log_lik <= tol).mean()),
                  "cmaes_physics_within_tol": bool(phys.delta_neg_log_lik <= tol),
                  "cmaes_evals_until_termination_physics": int(phys.n_evals),
                  "cmaes_evals_until_termination_random_median": float(rand.n_evals.median()),
                  "cmaes_evals_to_reach_tol_physics": phys.evals_to_reach_tol,
                  "cmaes_evals_to_reach_tol_random_median": float(np.nanmedian(rand.evals_to_reach_tol))}
    if sens_ode:
        f_glob_o = min(r.neg_log_lik for r in sens_ode)
        sens_tab["ode_start_neg_log_lik"] = [r.start_neg_log_lik for r in sens_ode]
        sens_tab["ode_neg_log_lik"] = [r.neg_log_lik for r in sens_ode]
        sens_tab["ode_delta_neg_log_lik"] = sens_tab["ode_neg_log_lik"] - f_glob_o
        sens_tab["ode_solves"] = [r.n_ode_solves for r in sens_ode]
        for i, n in enumerate(prior.names):
            sens_tab[f"ode_{n}"] = [r.theta[i] for r in sens_ode]
        summary_T2.update({
            "after_ode_fraction_random_within_tol": float((sens_tab.ode_delta_neg_log_lik.iloc[1:] <= tol).mean()),
            "after_ode_physics_within_tol": bool(sens_tab.ode_delta_neg_log_lik.iloc[0] <= tol),
            "ode_solves_physics": int(sens_tab.ode_solves.iloc[0]),
            "ode_solves_random_median": float(sens_tab.ode_solves.iloc[1:].median())})
    print(json.dumps(summary_T2, indent=1))
    sens_tab.to_csv(RUN_DIR / "8a_optimizer_sensitivity.csv", index=False)
    report.save_json(summary_T2, RUN_DIR / "8a_optimizer_sensitivity_summary.json")

    # plots: final values, convergence (switch CMA-ES -> curve_fit marked), trajectories of all starts
    fig, _ = report.plot_start_sensitivity(sens_tab, prior, f_glob, tol=tol)
    report.save_figure(fig, FIG_DIR / "8a_optimizer_sensitivity.png"); plt.show()
    fig, _ = report.plot_fit_histories(sens, sens_ode, title=f"{len(sens)} starts: CMA-ES (NN), then curve_fit (ODE)")
    report.save_figure(fig, FIG_DIR / "8a_optimizer_histories.png"); plt.show()
    fig = report.plot_trajectory_corner(prior, sens, sens_ode, truth=truth_disp,
                                        title=f"optimiser trajectories of {len(sens)} starts (red: physics-informed)")
    report.save_figure(fig, FIG_DIR / "8a_optimizer_trajectories_corner.png"); plt.show()
    final = sens_tab[[f"ode_{n}" for n in prior.names]].values if sens_ode else sens_tab[prior.names].values
    dcol = "ode_delta_neg_log_lik" if sens_ode else "delta_neg_log_lik"
    fig = report.plot_marginals_compare(
        {"within tol": prior.to_display(final[sens_tab[dcol].to_numpy() <= tol]),
         "not within tol": prior.to_display(final[sens_tab[dcol].to_numpy() > tol])},
        prior, truth=truth_disp, bins=25)
    report.save_figure(fig, FIG_DIR / "8a_optimizer_final_parameters.png"); plt.show()

In [ ]:
if RUN_SENSITIVITY:
    nc = sc["n_chains_per_group"]
    groups = {"A_dispersed": samplers.init_dispersed(D, nc, np.random.default_rng(21)),
              "B_best_fit": samplers.init_around(u_best, nc, np.random.default_rng(22))}
    group_seed = {"A_dispersed": 101, "B_best_fit": 102}
    sens_runs, sampler_g = {}, None
    for g, init_g in groups.items():
        (res_g, sampler_g), t_g = timeit_call(samplers.run_nuts, log_post, init_g, n_warmup=sc["n_warmup"],
                                              n_draws=sc["n_draws"], metric=nuts_cfg["metric"],
                                              target_accept=nuts_cfg["target_accept"], seed=group_seed[g],
                                              sampler=sampler_g, label=f"NUTS group {g}")
        log_timing(t_g, n_evals=res_g.info["evals_total_points"], n_grad_evals=res_g.info["evals_total_points"])
        sens_runs[g] = res_g
    # R-hat over all chains of both groups + quantile differences between the groups
    both = samplers.SamplerResult("both", theta=np.concatenate([r.theta for r in sens_runs.values()]),
                                  u=np.concatenate([r.u for r in sens_runs.values()]))
    diag_both = samplers.diagnostics(both, prior)
    qa = np.percentile(prior.to_display(sens_runs["A_dispersed"].flat_theta()), [16, 50, 84], axis=0)
    qb = np.percentile(prior.to_display(sens_runs["B_best_fit"].flat_theta()), [16, 50, 84], axis=0)
    chains_T2 = pd.DataFrame({"r_hat_all_chains": diag_both["r_hat"].values,
                              "q16_A": qa[0], "q16_B": qb[0], "median_A": qa[1], "median_B": qb[1],
                              "q84_A": qa[2], "q84_B": qb[2],
                              "z_median": (qa[1] - qb[1]) / (diag_both["sd"].values * np.sqrt(2.0 / diag_both["ess_bulk"].values))},
                             index=keys)
    display(chains_T2)
    chains_T2.to_csv(RUN_DIR / "8b_chain_sensitivity.csv")
    report.save_json({g: r.info for g, r in sens_runs.items()}, RUN_DIR / "8b_chain_sensitivity_info.json")
    fig = report.plot_marginals_compare({g: prior.to_display(r.flat_theta()) for g, r in sens_runs.items()},
                                        prior, truth=truth_disp)
    report.save_figure(fig, FIG_DIR / "8b_chain_sensitivity.png"); plt.show()

## 9. Profile likelihood (optional, T3 / R1 comment 5)
For a grid of fixed values of one parameter (default: $\log_{10}N_t$ of the dominant shallow trap) all other
parameters are re-optimised inside the prior support (batched L-BFGS from the best fit, random prior draws and,
in further sweeps, the neighbouring optima).  Plotted: $-2\Delta\log L$ with the $\chi^2_1$ thresholds 1 (68 %) and
3.84 (95 %), for the tr-PL-only and the joint likelihood (same $\sigma$s), together with the MCMC marginal of this run.
The difference between profile and marginal shows volume effects.  Optionally the endpoints of the 95 % range
are checked with the ODE.

In [ ]:
if RUN_PROFILE:
    pc = cfg["profile"]
    j = prior.names.index(pc["param"])
    prof_grid = np.linspace(prior.lo[j], prior.hi[j], pc["n_grid"])
    PROFILES, PROFILE_SUMMARY = {}, {}
    for mode in [m for m in pc["modes"] if m in LP and m != "sspl"]:
        if mode not in BEST_NN:                   # e.g. MODE='trpl' but joint profile requested
            run_fit(mode, np.vstack([BEST["trpl"], theta_start]), ["trpl_best", "physics"], mode, refine_ode=False)
        prof, t_prof = timeit_call(profile.profile_likelihood, LP[mode], pc["param"], prof_grid, BEST_NN[mode],
                                   n_random=pc["n_random"], n_sweeps=pc["n_sweeps"], verbose=False,
                                   label=f"profile {pc['param']} ({mode})")
        log_timing(t_prof, n_evals=prof.attrs["n_evals"], n_grad_evals=prof.attrs["n_evals"],
                   note=f"{pc['n_grid']} grid values")
        PROFILES[mode] = prof
        prof.to_csv(RUN_DIR / f"9_profile_{pc['param']}_{mode}.csv", index=False)
        s = {}
        for lev, thr in (("68", 1.0), ("95", 3.84)):
            lo_v, hi_v, lo_edge, hi_edge = profile.profile_interval(prof, thr)
            s[lev] = {"lo": lo_v, "hi": hi_v, "lo_at_prior_bound": lo_edge, "hi_at_prior_bound": hi_edge}
        # ODE check of the 95 % endpoints (do NN and ODE agree there?)
        if pc.get("ode_check_endpoints") and RUN_ODE_CHECK:
            for side in ("lo", "hi"):
                v = s["95"][side]
                if np.isfinite(v):
                    th_end = prof.loc[np.argmin(np.abs(prof.value - v)), prior.names].values.astype(float)
                    y_o = ode.ode_log10_tau(th_end, prior, npulse, art.constants, sur.qfls_used, T)
                    ok = np.isfinite(y_o)
                    y_n = LP["trpl"].predict_trpl(th_end)["log10_tau_used"][0]
                    s["95"][f"{side}_mse_trpl_nn"] = float(np.mean((y_n - term_trpl.y_obs) ** 2))
                    s["95"][f"{side}_mse_trpl_ode"] = float(np.mean((y_o[ok] - term_trpl.y_obs[ok]) ** 2))
        PROFILE_SUMMARY[mode] = s
    print(json.dumps(PROFILE_SUMMARY, indent=1))
    report.save_json(PROFILE_SUMMARY, RUN_DIR / "9_profile_summary.json")

    fig, ax = plt.subplots(figsize=(6.5, 4.2))
    for i, (mode, prof) in enumerate(PROFILES.items()):
        report.plot_profile(prof, prior, mcmc_samples_disp=samples_nuts if mode == MODE else None,
                            label=f"profile ({mode})", ax=ax, color=f"C{i}")
    report.save_figure(fig, FIG_DIR / f"9_profile_{pc['param']}.png"); plt.show()

## 10. tr-PL only vs joint (optional, T4 / R1 comment 5)
Needs a finished run of the other mode (`COMPARE_WITH`).  Both runs use the same prior, bounds and
$\sigma_\mathrm{trpl}$; each has its own optimisation and sampling.  Reported: overlaid marginals and the ratio of the
68 % / 95 % widths per parameter and for the Eq. 10/11 combinations.

In [ ]:
if COMPARE_WITH:
    other_dir = Path(COMPARE_WITH)
    other_settings = json.load(open(other_dir / "settings.json"))
    other_mode = other_settings["mode"]
    other = pd.read_csv(other_dir / "posterior_samples_nuts.csv")
    theta_other = other[prior.names].values
    runs = {MODE: nuts_res.flat_theta(), other_mode: theta_other}
    if other_mode == MODE:
        raise ValueError(f"COMPARE_WITH must be a run of the other mode (this run: {MODE}, other: {other_mode})")
    a, b = "trpl", "joint"
    ci_runs = {m: report.credible_intervals(prior.to_display(t), prior.display_bounds(), keys) for m, t in runs.items()}
    der_runs = {m: report.derived_intervals(t, prior) for m, t in runs.items()}
    widths = pd.concat([report.compare_widths(ci_runs[a], ci_runs[b], a, b),
                        report.compare_widths(der_runs[a], der_runs[b], a, b)])
    display(widths)
    widths.to_csv(RUN_DIR / "10_widths_trpl_vs_joint.csv")
    fig = report.plot_marginals_compare({m: prior.to_display(t) for m, t in runs.items()}, prior, truth=truth_disp)
    report.save_figure(fig, FIG_DIR / "10_marginals_trpl_vs_joint.png"); plt.show()

## 11. Timing and evaluation summary (T8 / R1 comment 7, R2 comment 2)
All `timeit` results of this run with the number of likelihood evaluations (rows = parameter sets) and of
evaluations that included a gradient.  The NUTS evaluation count includes the warm-up and the padding of
chains that finished their trajectory early (batched execution); `leapfrogs used` in the note counts only
the gradient evaluations the chains actually used during sampling.  The **T8 table** combines the offline costs
(training simulations and network training, read from the training records) with the per-sample costs of this run
and the corresponding costs with the rate equations (measured where run, otherwise evaluations x measured cost
per ODE solve; NUTS with the ODE would need finite-difference gradients).

**Mean ± SD**: every measured step is repeated with `timeit` (`TIMING_REPEATS` for the short steps,
`TIMING_REPEATS_LONG` for NUTS, emcee, the grid and the ODE-only CMA-ES; *n* in the table).  One-time costs
are excluded everywhere: the XLA programs are compiled beforehand (NUTS: in a short untimed run whose
duration is given in the note), and the short steps get one untimed warm-up call before their repeats.  ODE estimates are
evaluations × (mean ± SD of one timed ODE solve at the best fit), so their ± is the spread of the solve time only.
The offline rows are single runs; for the simulations the mean ± SD per simulation is given.  The table
also has numeric columns (`nn_mean_s`, `nn_sd_s`, `n_repeats`, `ode_mean_s`, `ode_sd_s`) for combining runs.

**Sampler efficiency** (NUTS and emcee rows of the T8 table, and `11_sampler_efficiency.csv` with one row per sampler for the
paper table; empty for the grid): `min_ess_bulk`, `min_ess_tail` and `max_r_hat` over the free parameters;
`ess_per_s` = min ESS bulk / timed total (timeit mean of the timed run, warm-up/burn-in included; for NUTS it also
includes the XLA (re)compilation inside the timed run, `compile_s` = extra time of its first chunk, while the first
compilation happens in the short untimed run beforehand, see the note); `ess_per_s_sampling` = min ESS bulk /
sampling phase (NUTS only; emcee records only the total time); `ess_per_1k_evals` = 1000 × min ESS bulk / likelihood
evaluations (hardware-independent; a NUTS evaluation includes the gradient). NUTS only: `step_size`, `mean_leapfrogs`,
`n_divergent`, `compile_s`, `time_warmup_s`, `calls_per_iteration` = batched calls / (warm-up + draws) — larger than
`mean_leapfrogs` because chains that finish their trajectory early wait for the longest one (batch padding).
emcee only: `max_tau_int` (integrated autocorrelation time, steps), `acceptance_fraction`.

**Rows added later** by `smc.ipynb` and `cylinder.ipynb` (`benchmark.update_t8`, run on this folder): tempered SMC
(min ESS over the islands, R-hat, log Z; `ess_kish`) and the eigenvector-cylinder importance sampling (Kish ESS; its
σ_trPL is the cylinder σ, given in the row).  **Mode `sspl`**: no surrogate is involved (the ss-PL model is exact), so
the ODE column equals the NN column for the samplers; the optimisation row sums every fit the mode needs.

In [ ]:
timing_table = pd.DataFrame(TIMINGS)[["label", "repeats", "warmup", "best_s", "mean_s", "std_s", "n_evals", "n_grad_evals", "note"]]
timing_table.to_csv(RUN_DIR / "11_timing.csv", index=False)
gpu = tf.config.experimental.get_device_details(tf.config.list_physical_devices("GPU")[0]).get("device_name", "?") \
      if tf.config.list_physical_devices("GPU") else "CPU"

# ── T8 table: offline costs, per-sample costs with the NN, and the ODE equivalents ──────────
# Measured steps: mean ± SD over the timeit repeats (n = repeats).  ODE estimates: number of
# evaluations x (mean ± SD of one timed ODE solve) — their ± is the spread of the solve time only.
fmt = benchmark.format_mean_sd
costs = benchmark.offline_costs(nn_dir, art.paths["training_hdf5"])
ode_s = ODE_CHECK.get("ode_seconds_per_solve") if ODE_CHECK else None
ode_sd = ODE_CHECK.get("ode_seconds_per_solve_sd", 0.0) if ODE_CHECK else 0.0

def t_of(*prefixes):
    """Timed steps whose label starts with one of the prefixes, summed: (mean, SD, repeats, evaluations)."""
    rows = [r for r in TIMINGS if r["label"].startswith(prefixes)]
    if not rows:
        return None
    return (sum(r["mean_s"] for r in rows), float(np.sqrt(sum(r["std_s"] ** 2 for r in rows))),
            min(r["repeats"] for r in rows), sum(int(r["n_evals"] or 0) for r in rows))

def ode_estimate(n_solves, note="estimated (serial)"):
    """ODE-workflow cost of n_solves serial ODE solves: (text, mean_s, sd_s)."""
    if not ode_s:
        return "—", None, None
    return (f"{fmt(n_solves * ode_s, n_solves * ode_sd, None, 'h', prefix='≈ ')} {note}",
            n_solves * ode_s, n_solves * ode_sd)

t8 = []
def add(stage, step, nn_text, ode_text, nn=None, ode=(None, None), extra=None):
    t8.append({"stage": stage, "step": step, "nn_workflow": nn_text, "ode_workflow": ode_text,
               "nn_mean_s": nn[0] if nn else None, "nn_sd_s": nn[1] if nn else None,
               "n_repeats": nn[2] if nn else None, "ode_mean_s": ode[0], "ode_sd_s": ode[1], **(extra or {})})

# sampler efficiency (NUTS / emcee rows; grid and the other rows stay empty): min ESS and max R-hat over the
# FREE parameters, ESS per second of the timed run and per 1000 likelihood evaluations (NUTS: with gradient)
FREE_KEYS = [prior.display_keys[i] for i in prior.free_idx]
EFF = []

sim_txt = (f"{costs.get('simulation_cpu_h', float('nan')):.2f} CPU-h "
           f"({costs.get('simulation_wall_h', float('nan')):.2f} h wall)")
if costs.get("simulation_s_per_solve_mean"):
    sim_txt += (f"; {fmt(costs['simulation_s_per_solve_mean'], costs['simulation_s_per_solve_sd'] or 0.0, None, 'ms')}"
                " per simulation")
add("offline", f"training simulations ({costs.get('n_simulations')} ODE solves)", sim_txt, "—")
add("offline", f"network training ({costs.get('epochs_trained')} epochs)",
    f"< {costs.get('training_wall_h_upper', float('nan')):.2f} h (GPU, single run)", "—")
for prefix, name in (("tr-PL preprocessing", "preprocessing"), ("start-point", "physics-informed start values")):
    r = t_of(prefix)
    if r:
        add("per sample", name, fmt(r[0], r[1], r[2], "s"), fmt(r[0], r[1], r[2], "s"), nn=r[:3], ode=r[:2])
if ode_s:
    add("per sample", "one ODE solve at the best fit (basis of the ODE estimates)", "—",
        fmt(ode_s, ode_sd, ODE_CHECK["ode_solve_repeats"], "ms"), ode=(ode_s, ode_sd))
# optimisation: every fit this mode needs (sigma_eff of each data set; the tr-PL best fit is a start of the
# ss-PL / joint fits).  The ss-PL-only fit uses the exact ss-PL model, i.e. it costs the same in both workflows.
FIT_TAGS = {"trpl": ["tr-PL only"], "sspl": ["tr-PL only", "ss-PL only"],
            "joint": ["tr-PL only", "ss-PL only", "joint"]}[MODE]
fits_t = {tag: t_of(f"fit {tag}: CMA-ES") for tag in FIT_TAGS}
fits_t = {k: v for k, v in fits_t.items() if v}
r = (sum(v[0] for v in fits_t.values()), float(np.sqrt(sum(v[1] ** 2 for v in fits_t.values()))),
     min(v[2] for v in fits_t.values()), sum(v[3] for v in fits_t.values()))
r_ode = t_of("fit tr-PL with ODE") if MODE == "trpl" else None
if r_ode:
    ode_txt, ode_num = (f"{fmt(r_ode[0], r_ode[1], r_ode[2], 'min')} measured "
                        f"(1 start, {r_ode[3]:,} solves, {cfg['ode']['n_jobs']} workers on {os.cpu_count()} logical CPUs)"), r_ode[:2]
else:
    ss = fits_t.get("ss-PL only")
    ode_txt, *ode_num = ode_estimate(r[3] - (ss[3] if ss else 0))
    if ss and ode_num[0] is not None:
        ode_num = [ode_num[0] + ss[0], float(np.hypot(ode_num[1], ss[1]))]
        ode_txt = (f"{fmt(ode_num[0], ode_num[1], None, 'h', prefix='≈ ')} estimated (serial; "
                   f"ss-PL-only fit as measured: {fmt(ss[0], ss[1], ss[2], 's')})")
fit_names = "" if len(fits_t) == 1 else "; fits: " + " + ".join(fits_t)
add("per sample", f"optimisation (CMA-ES, all starts{fit_names})", f"{fmt(r[0], r[1], r[2], 's')}; {r[3]:,} evaluations",
    ode_txt, nn=r[:3], ode=tuple(ode_num))
r = t_of("fit tr-PL only: curve_fit", "fit joint: curve_fit")
if r:
    txt = f"{fmt(r[0], r[1], r[2], 's')}; {r[3]:,} ODE solves, parallel"
    add("per sample", "local refinement curve_fit (ODE in both workflows" +
        ("; tr-PL fit = start of the ss-PL fit)" if MODE == "sspl" else ")"), txt, txt, nn=r[:3], ode=r[:2])
for prefix, name in (("NUTS (warm-up", "posterior sampling: NUTS"), ("emcee", "posterior sampling: emcee"),
                     ("grid (", "grid over the full prior box" if GRID_RANGE == "prior" else "grid (window around the fit)")):
    r = t_of(prefix)
    if not r:
        continue
    if MODE == "sspl":              # no surrogate: the exact ss-PL model is the same computation in both workflows
        ode_txt, ode_num = "same as NN workflow (exact ss-PL model, no surrogate)", list(r[:2])
    elif prefix.startswith("NUTS"):   # ODE has no gradient: finite differences cost (D_free + 1) solves per leapfrog
        lf = nuts_res.info["leapfrogs_sampling"] * (1 + nuts_res.info["n_warmup"] / nuts_res.info["n_draws"])
        ode_txt, *ode_num = ode_estimate(lf * (len(prior.free_idx) + 1), "estimated (FD gradients, serial)")
    else:
        ode_txt, *ode_num = ode_estimate(r[3])
    eff = None
    if prefix.startswith("NUTS"):
        eff = benchmark.sampler_efficiency("nuts", diag_nuts, nuts_res.info, r[0], FREE_KEYS)
    elif prefix == "emcee" and emcee_res is not None:
        eff = benchmark.sampler_efficiency("emcee", diag_emcee, emcee_res.info, r[0], FREE_KEYS)
    nn_txt = f"{fmt(r[0], r[1], r[2], 'min')}; {r[3]:,} evaluations"
    if eff:
        nn_txt += f"; {benchmark.efficiency_text(eff)}"
        EFF.append(eff)
    add("per sample", name, nn_txt, ode_txt, nn=r[:3], ode=tuple(ode_num),
        extra={k: v for k, v in (eff or {}).items() if k not in ("sampler", "timed_mean_s", "n_evals")})
t8 = pd.DataFrame(t8)
t8.to_csv(RUN_DIR / "11_T8_timing_table.csv", index=False)
sampler_eff = pd.DataFrame(EFF, columns=benchmark.EFFICIENCY_COLUMNS)      # one row per sampler (paper table)
sampler_eff.to_csv(RUN_DIR / "11_sampler_efficiency.csv", index=False)
report.save_json({"timings": TIMINGS, "device": gpu, "tensorflow": tf.__version__, "offline_costs": costs,
                  "ode_seconds_per_solve": ode_s}, RUN_DIR / "11_timing.json")
print("device:", gpu, "| ODE cost per solve [s]:", ode_s, "±", ode_sd)
pd.set_option("display.max_colwidth", 200)
display(t8[["stage", "step", "nn_workflow", "ode_workflow"]])
display(sampler_eff)
timing_table